Title: PRAGMA: Revolut Foundation Model

URL Source: https://arxiv.org/pdf/2604.08649

Published Time: Mon, 13 Apr 2026 00:06:54 GMT

Number of Pages: 21

Markdown Content:
# PRAGMA: Revolut Foundation Model 

Maxim Ostroukhov 1 Ruslan Mikhailov 1 Vladimir Iashin 1

Artem Sokolov 1 Andrei Akshonov 1 Vitaly Protasov 1 Dmitrii Beloborodov 1

Vince Mullin 2 Roman Y. Enzmann 2 Georgios Kolovos 2 Jason Renders 2

Pavel Nesterov 1 Anton Repushko 1

> 1

Revolut Research 2NVIDIA 

Modern financial systems generate vast quantities of transactional and event-level data that encode rich economic signals. This paper presents PRAGMA, a family of foundation models for multi-source banking event sequences. Our approach pre-trains a Transformer-based architecture with masked modelling on a large-scale, heterogeneous banking event corpus using a self-supervised objective tailored to the discrete, variable-length nature of financial records. The resulting model supports a wide range of downstream tasks such as credit scoring, fraud detection, and lifetime value prediction: strong performance can be achieved by training a simple linear model on top of the extracted embeddings and can be further improved with lightweight fine-tuning. Through extensive evaluation on downstream tasks, we demonstrate that PRAGMA achieves superior performance across multiple domains directly from raw event sequences, providing a general-purpose representation layer for financial applications. 

Disclaimer : We report only relative improvements, as absolute metrics are commercially sensitive. All examples are synthetic and not from real production data. Credit Recurrent Trx. LTV Comms Fraud Product Rec.    

> Task-specific model PRAGMA-S (10 M) PRAGMA-M (100 M) PRAGMA-L (1 B)

Figure 1: A single architecture from 10M to 1B parameters that outperforms task-specific models across tasks. 

## 1 Introduction 

Foundation models are general-purpose models trained at scale on broad data distributions and subsequently adapted to a wide variety of downstream tasks (Bommasani et al., 2021). While such models have transformed natural language processing (Devlin et al., 2019; Brown et al., 2020) and computer vision (Kirillov et al., 2023; Caron et al., 2021), their application to multi-source banking user histories remains comparatively underexplored. Modern banks and fintechs accumulate large volumes of data: event streams spanning card and transfer transactions, product usage, in-app navigation, and customer communications, alongside static generalised profile state such as account tenure and 1

> arXiv:2604.08649v1 [cs.LG] 9 Apr 2026

plan. These event streams encode signals relevant to risk management, product analytics, and operations, but they are difficult to model efficiently with off-the-shelf language-model tokenisation and architectures. While serialising structured records as text and feeding them to a standard Transformer is a viable baseline, it inflates sequence lengths considerably because every field name and delimiter becomes several subword tokens. Moreover, numerical values are split into digit fragments that discard magnitude and ordering, both of which are critical for financial reasoning. Together, these limitations make naive text serialisation impractical for the long, heterogeneous user histories common in banking. Multi-source banking user histories differ from text in three ways. First, each event is a variable-length record with mixed categorical, numerical, and free-text fields. Second, histories are long-tailed in length and irregular in time, with strong daily and weekly cycles. Third, practical deployments must operate under strict privacy and regulatory constraints, which limit what can be reported and which features can be used for certain decisions. Because no single off-the-shelf architecture handles all three challenges simultaneously, practitioners default to building task-specific pipelines with extensive feature engineering, making it hard to share statistical strength across domains and products. Prior work addresses isolated slices of this problem. Tabular Transformers such as TabTransformer and FT-Transformer (Huang et al., 2020; Gorishniy et al., 2021) model fixed-schema rows, while sequential recommender models such as SASRec and BERT4Rec (Kang et al., 2018; Sun et al., 2019) operate on item-like interaction histories. Financial foundation models have largely focused on text or generic time-series tokenisation (Yang et al., 2020; Wu et al., 2023; Yang et al., 2023; Jin et al., 2024; Ansari et al., 2024), while newer transaction-ledger models such as nuFormer and TransactionGPT (Braithwaite et al., 2025; Dou et al., 2025) move closer to our setting. However, these models typically ingest a single event source, omit static profile state, and are evaluated on a narrow set of tasks: nuFormer targets product recommendation, while TransactionGPT focuses on anomaly detection and trajectory gener-ation. The literature still lacks a multi-source encoder backbone with explicit profile state that transfers across a broad range of discriminative banking tasks. In this paper, we present PRAGMA, a family of encoder-style foundation models for multi-source banking user his-tories. PRAGMA is pre-trained with masked modelling on a large-scale corpus of user histories that combines multi-source events with static profile state (§2.1). To handle heterogeneity, we apply a key–value–time tokenisation scheme with type-specific value encoding for numerical, categorical, and textual fields (§2.2). The resulting backbone uses two encoder branches for profile state and events whose outputs are fused by a history encoder (§2.3). We choose an encoder-only, bidirectional design because our primary goal is transferable representations for dis-criminative financial tasks, rather than open-ended generation. Masked modelling enables each token to attend to both past and future context (Devlin et al., 2019), which is particularly useful when reconstructing partially observed event records and learning record-level representations from complete histories. After pre-training, PRAGMA can be adapted efficiently in two complementary ways (§3.1). In the embedding probe setting, we freeze the backbone and train a lightweight head on top of the extracted embeddings. In the LoRA fine-tuning setting, we apply Low-Rank Adaptation (LoRA) (Hu et al., 2022) to update only a small fraction of parameters, enabling fast specialisation while keeping most of the backbone shared across tasks. We evaluate PRAGMA on a suite of internal downstream benchmarks spanning credit scoring, fraud detection, com-munication engagement, recurrent transaction detection, lifetime value prediction, and more (§3.2). Across evaluated domains, PRAGMA consistently outperforms strong task-specific baselines while reducing the need for hand-crafted features (Figure 1). We further describe the engineering choices required to train PRAGMA efficiently on long and highly variable user histories, including sequence packing and dynamic batching (§2.4). Our contributions are as follows: • We introduce PRAGMA, a family of encoder-style foundation models for multi-source banking user histories, scaling from 10 M to 1 B parameters, to our knowledge, the largest published encoder backbone for consumer banking event sequences. The architecture combines a key–value–time tokenisation scheme with a two-branch design in which profile-state and event encoders feed a history encoder for heterogeneous financial records. • We describe an efficient pre-training recipe for long and irregular banking user histories based on masked modelling, sequence packing, and dynamic batching, and show that LoRA fine-tuning of a pre-trained back-bone consistently matches or outperforms full training from scratch. 2Created  24 �04 �07 19:20:18                                                      

> Type communication
> Channel email
> Product stocks_shares_isa
> Interact interacted
> Created 20 �11 �02 12:09:04
> Type topup
> Direction in
> Amount 100.00
> Currency gbp
> Fee 0.00
> ... ...
> Created 20 �11 �02 12:19:54
> Type card_payment
> Direction out
> Amount 14.99
> Currency gbp
> Fee 0.00
> Description metal plan
> MCC 6012
> ... ...
> Created 22 �12 �25 09:13:48
> Type p2p_transfer
> Direction out
> Amount 150.00
> Currency eur
> Fee 0.00
> ... ...
> Created 22 �12 �25 09:13:40
> Type app_event
> View con �rm_p2p_dialog
> Created 22 �12 �25 09:13:35
> Type app_event
> View p2p_amount
> Created 24 �04 �07 19:24:02
> Type trading
> Direction buy
> Symbol swda
> Amount 100
> Price 74.1945
> Currency gbx
> Order type market
> ... ...
> Region uk
> Plan metal
> Balance 6,012.54
> Currency gbp
> Age 25–35
> Device iphone18,2
> ... ...
> Evaluation Point
> Created 25 �07 �22 13:49:29
> Type app_event
> Screen junior_transfer

Figure 2: Event timeline overview . After account creation, users generate a sequence of platform interactions over time, spanning transactions, in-app navigation, and communications. We aggregate the event history up until a designated evaluation point. Alongside these sequential events, we capture contextual attributes that describe the record’s state at that point, e.g., membership plan or service region. Both events and attributes share a uniform representation: a timestamp and a set of key–value pairs (e.g., Type: card_payment , Channel: email ). All values shown are synthetic; the figure is for illustration purposes only. • We evaluate a single pre-trained backbone across six diverse downstream tasks (credit scoring, fraud detec-tion, lifetime value, communication engagement, recurrent transaction detection, and product recommenda-tion), a substantially broader task scope than prior transaction-ledger models, which typically target one or two tasks. PRAGMA consistently outperforms strong task-specific baselines while reducing the need for hand-crafted features. 

## 2 Pre-training 

2.1 Dataset 

Our goal is to build a foundation model that encodes diverse event-level signals and transfers across a wide range of downstream tasks. Our dataset is structured at the record level, where each observation represents a pseudonymised event history associated with an evaluation point. As shown in Figure 2, we consider an event history alongside contextual attributes. This approach enables the model to account for both sequential patterns and time-invariant features like account currency. All data used in this work is fully anonymised and contains no personally identifiable information. We construct our pre-training dataset from 26 M user records spanning 111 countries, accumulating 24 B events that total 207 B tokens. 

2.1.1 Event History 

Standard platform usage generates event streams across various services, e.g., account funding, payments, in-app navigation, or service communications. These aggregated event histories capture population-level patterns that support a range of analytical and predictive tasks. An event is defined by a created timestamp and a set of key–value pairs, e.g., Direction: out . We fetch events from broad source types that can be loosely grouped into transactions, app, trading, and communication, which were selected for their high expected impact on downstream tasks. Event schemas are specific to their source type and incorporate distinct sets of keys, e.g., Symbol key is unique to trading events. Beyond anonymisation, de-identification, and standard eligibility criteria, no additional statistical filtering or 3pre-processing, such as outlier removal or vocabulary pruning, is applied to the event streams, to ensure that the model captures the full heterogeneity found in production. 

2.1.2 Profile State 

In addition to the event history, we incorporate general contextual attributes such as balance quantile, plan, insurance state, and service region. These attributes provide useful context that is otherwise missing from the event history alone. Profile state is a set of descriptive key–value pairs in an event-like format, e.g., Plan: metal , timestamped at the designated evaluation point (or the cut-off date during pre-training). High-activity users often generate tens of thousands of interactions, exceeding computational bounds; we address this via truncation to a fixed context window (§2.3.5). However, truncation risks discarding early historical milestones that carry useful signal, such as account age. We therefore augment profile state with life-long events , key–value pairs that, unlike regular profile attributes, each carry an individual timestamp recording a first occurrence, e.g., Lifelong: first_topup at 20-11-02 12:09:04 . This timestamp is then used to compute the temporal distance to the evaluation point, enabling the model to encode the timing of historical milestones. 

2.1.3 Pre-training Time Range 

Developing a robust and generalisable model requires a delicate balance between maximising historical coverage and maintaining data relevance. Accordingly, determining the optimal temporal range for pre-training involves navigating several trade-offs between event diversity, distribution shift, and computational efficiency. First, simply including every event from the full available dataset is often impractical and sub-optimal. Older events may reflect historical patterns, product features, or system dynamics that are no longer relevant at inference time. Such discrepancies create a distribution mismatch that can degrade performance, as the model may struggle to generalise from obsolete historical examples to the evolving behaviours present in deployment. Additionally, the inclusion of highly heterogeneous events from long time spans can make the pre-training task harder and slow down model con-vergence. Second, downstream applications may require making predictions on events that took place within temporal ranges either much earlier or much later than those used for pre-training. If the model is not exposed to sufficient diversity in both recent and less-common historical patterns, the performance on these out-of-distribution inputs may suffer. Finally, Transformer architectures have a limited effective context span, determined both by model design and hardware constraints. With these considerations in mind, we select a temporal range of 25 months from 2023 to 2025 for pre-training, balancing comprehensive event coverage, recency, distribution consistency, and tractable sequence modelling. 

2.2 Tokenisation 

Unlike standard LLMs that treat everything as text, a financial foundation model needs to preserve the structural nature and heterogeneity of tabular data. We address this challenge by implementing a disentangled embedding space of input tokens. As shown in Figure 3, we represent each data point by three components: a semantic type (key), a value, and a temporal coordinate, following a common standard in tabular event data (Braithwaite et al., 2025). For instance, Channel: email at 24-04-07 19:20:18 maps to a key, a value, and a temporal coordinate, respectively. This ensures that the model distinguishes between the meaning of a field and its value, while also encoding event chronology. Next, we present how the three are tokenised. 

Semantic Type (Key). The semantic type embedding enables the model to learn the meaning of a field and to contextualise the value it holds. We tokenise all semantic types (keys) as single tokens, and both event and profile state semantic types are encoded in a similar way. This results in a vocabulary of ∼60 tokens. 

Value. We cover the diversity of values with three value types: numerical , categorical , and textual . Numerical values are mapped to percentile buckets, where bin boundaries are learned from training data with an extra bucket for zero, allocating one token per bucket. The distinction between categorical and textual is determined by cardinality thresholding: string fields whose number of unique values falls below a predefined threshold are treated as categorical, 4Created  20 �11 �02 12:19:54                                            

> Type card_payment
> Direction out
> Amount 14.99
> Currency gbp
> Fee 0.00
> Description metal plan
> MCC 6012
> ... ...
> 20 �11 �02 12:19:54 Type Direction Amount Currency Fee Description MCC
> card_payment out 14.99 gbp 0.00 6012
> perc_24 perc_0
> category category number category number text category
> 27021 2892 7688 2259 7645 26965
> 6148 5729 9153 11847 9740 8964 739
> Keys
> Values
> 18.95
> log-seconds
> to the last event
> 12 12
> Timestamp
> day
> hour
> week
> day
> mth
> day
> metal plan
> met plan
> 736 1146
> al
> 2687

Figure 3: Tokenisation overview . A raw event record is decomposed into a temporal coordinate, semantic types (keys), and values. Keys are always represented by one token, while values use type-specific tokenisation: numerical values are bucketised by percentile, categorical values map to a single token, and textual values are split into subword tokens. Some keys therefore expand to multiple value tokens, e.g., Description → met , al , plan . Time is encoded both as log-seconds to the last event and as calendar and time features derived from the timestamp. Profile state is encoded similarly to an event record. while higher-cardinality fields are treated as textual. Categorical values are manually selected from all text fields to prevent splitting common values, such as merchant category codes (MCC), into multiple tokens, and are represented as a single token as well. For textual fields, values are tokenised with a BPE-style subword tokeniser (Sennrich et al., 2016) with a reserved [UNK] token for rare unseen fragments. In total, values allocate a vocabulary of ∼28 k tokens. 

Temporal Information. We encode time in two ways. First, we compute the elapsed time since the most recent event, measured in seconds. We then apply a soft logarithmic transformation, 8 · ln(1 + t/ 8) , to compress the dynamic range of life-long events while preserving high-resolution linear granularity for recent events. This prevents aliasing in positional embeddings caused by extreme temporal gaps without sacrificing the precision of local event sequenc-ing. Second, to capture daily and weekly temporal cycles, we additionally decompose each event timestamp into its cyclical constituents: hour of day, day of week, and day of month, and embed them using periodic functions similar to Gorishniy et al. (2022), but with periods fixed to the known calendar cycles rather than learned. Calendar features are applied only to event-history entries, as cyclical patterns are less relevant for one-off life-long events where the log-seconds encoding already captures the relevant temporal signal. 

2.3 Model Architecture 

PRAGMA is an encoder-only Transformer that inputs an event history along with contextual attributes and outputs dense record-level embeddings. It is trained on a large-scale, diverse dataset with a masked modelling (MLM) objec-tive that reconstructs masked input tokens. Once pre-trained, it acts as a backbone for downstream adaptation with small-scale (2–4 % of the model’s parameters) fine-tuning for a variety of tasks. An overview of PRAGMA is shown in Figure 4. PRAGMA is parametrised as a family of models with 10 M, 100 M, and 1 B parameters, enabling selection according to operational budget and constraints. The details of the architecture family are provided in Table 1. All size variants use GELU activations (Hendrycks et al., 2016), pre-norm layer normalisation (Xiong et al., 2020), and dropout of 0.1 (Srivastava et al., 2014). The model consists of three main blocks: Profile State Encoder, Event Encoder, and History Encoder. First, the profile state tokens are processed by the Profile State Encoder. Second, similar to profile state, each event is encoded independently in the Event Encoder. Finally, the outputs of the Profile State and Event Encoders are concatenated and encoded in the History Encoder to form an output. Depending on the stage, the final output is used either in an MLM head during pre-training, a classification head during fine-tuning, or as-is in an embedding probe. 5USR      

> Emb
> USR
> Emb
> Pro �le State
> Encoder
> RoPE
> EVT
> Emb
> EVT
> Emb
> Event Encoder
> EVT
> EVT
> EVT
> EVT
> DATE
> DATE
> DATE
> EVT
> EVT
> USR
> EVT
> USR
> EVT
> EVT
> EVT
> Proj
> History Encoder
> RoPE
> Time to
> Last Event
> Values
> Keys
> Timestamps
> Time since
> Life-long Events
> Values
> Keys
> Event #1 Event #2 Event #3
> Events
> Pro �le State

Figure 4: PRAGMA backbone overview . Each user record is represented as an ordered event history and profile state, where every field is decomposed into a semantic type (key), one or more values, and a temporal coordinate. Keys and values are embedded from a shared lookup table, and value tokens receive within-field positional embeddings. A 

Profile State Encoder maps profile state xa, with time since life-long events ta encoded via RoPE, into a [USR] 

embedding za, while an Event Encoder independently maps the tokens of each event xe into a [EVT] embedding z′

> e

and adds calendar features zt. A History Encoder then contextualises the sequence z = [ za : ze] with time to the last event te encoded via RoPE, producing a representation for a user record zh.

Width Depth Model Params dmodel dffn Profile Event History Heads 

PRAGMA-S 10 M 192 768 1 5 2 3PRAGMA-M 100 M 512 2048 3 16 6 8PRAGMA-L 1 B 1024 4096 9 45 18 16 Table 1: PRAGMA model family . PRAGMA scales across three variants (10 M, 100 M, 1 B parameters) by jointly increasing model width ( dmodel , dffn ), depth of the profile-state, event, and history encoders, and the number of attention heads. 

2.3.1 Token Embedding 

Profile state and event tokens are embedded identically. For multi-valued fields (e.g., Description ), the key token is replicated to match each of its values, yielding n key–value pairs in total. A single shared embedding table E maps each key and value to a d-dimensional vector; the two embeddings are summed and augmented with static sine/cosine positional encodings (PosEmb) (Vaswani et al., 2017): 

x = PosEmb �E(k) + E(v), x ∈ Rn×d. (1) Positions index values within a field, not across fields—e.g., the value eur of Currency receives position 0, while the three value tokens (met, al, plan) of Description receive positions (0, 1, 2) (see Figure 3). We denote user and event embeddings as xa ∈ Rna×d and xe ∈ Rne×d, respectively. Following common practice in encoder-only Transformers (Devlin et al., 2019; Dosovitskiy et al., 2021), a learnable [USR] (or [EVT] ) token is prepended to each sequence (Figure 4). 

2.3.2 Profile State Encoder 

The Profile State Encoder is a bidirectional Transformer. It inputs the profile state tokens xa ∈ Rna×d and correspond-ing temporal coordinates ta ∈ Rna , where each entry holds the log-seconds since the corresponding life-long event (or 0 for non-life-long pairs). We use RoPE (Su et al., 2024) to encode ta. We disentangle this positional embedding 6from the value-level positional embedding discussed in §2.3.1 to avoid the semantic and scale mismatch. The output is a sequence of profile state embeddings za ∈ Rna×d. We pass the first element, which corresponds to the [USR] 

token, to the History Encoder—we refer to it as za ∈ R1×d for simplicity. 

2.3.3 Event Encoder 

The Event Encoder is a bidirectional Transformer, similar to the Profile State Encoder. It inputs an event history 

xe = ( xe, 1, x e, 2, . . . , x e,n e ), where each element has a distinct number of token embeddings ( xe,i ∈ Rni×d), and processes each event independently of all other events in the history. The module outputs a token-level embedding sequence for each event, denoted bze, which is used by the MLM head during pre-training. Similar to the Profile State Encoder, we select the first token corresponding to the [EVT] token for each event as its aggregated representation 

z′ 

> e

∈ Rne×d.The calendar features (hour of day, day of week, and day of month) xt ∈ Rne×3 are converted to sine and cosine radians and embedded with two MLP layers into zt ∈ Rne×d. Next, the embedded calendar features are added to the Event Encoder output: ze = z′ 

> e

+ zt.

2.3.4 History Encoder 

The History Encoder is a bidirectional Transformer, similar to the other two encoders. It inputs the concatenated aggregated representations of profile state and the calendar-augmented events: z = [ za : ze] ∈ R(1+ ne)×d, as well as the corresponding temporal coordinate te ∈ R1+ ne , where each entry holds the log-seconds to the most recent event in the history ( 0 for the za position). Similar to the Profile State Encoder, RoPE is used to encode positional information. The output is a sequence of embeddings zh ∈ R(1+ ne)×d, where zh, 0 corresponds to [USR] and zh, 1, . . . , z h,n e to the [EVT] tokens. zh is used by the MLM head during pre-training and for downstream probes. 

2.3.5 Training Pre-training Objective. PRAGMA is pre-trained with an MLM objective following BERT (Devlin et al., 2019) where a random subset of event input tokens is masked, and the model reconstructs the original tokens. For each masked token, the MLM head receives the concatenation of three d-dimensional vectors: the Event Encoder output at that token’s position within bze, providing local within-event context; the History Encoder output at the corresponding 

[EVT] position zh,i , providing cross-event context; and the History Encoder output at the [USR] position zh, 0,providing user-level context. This 3d-dimensional representation is projected back to d dimensions and matched against the embedding table to produce logits. The training loss is cross-entropy with label smoothing (Szegedy et al., 2016). 

Masking Strategy. The masking strategy combines three sources: standard individual token-level masking (with 15 % probability), event-level masking (10 %) that requires the model to reconstruct an entire event, and semantic-type (key)-level masking (10 %) where all values of the selected keys are masked, training the model to predict values given context and a key. During pre-training, a small fraction of selected positions are replaced with [UNK] rather than 

[MASK] . Because [UNK] positions are excluded from the MLM objective, they receive no gradient and effectively act as a form of input dropout, training the model to recover original values under a stronger corruption scheme and reducing reliance on the presence of [MASK] , which does not occur at inference time. 

Downstream Adaptation. PRAGMA supports two modes of downstream adaptation. In the embedding probe mode , the record-level representation produced by the History Encoder is extracted as a frozen feature vector, and a lightweight linear probe is trained on top. In the LoRA fine-tuning mode , a small fraction ( ∼2–4 %) of model weights (the attention and feed-forward projections) are updated via Low-Rank Adaptation (Hu et al., 2022), keeping the pre-trained backbone mostly frozen and reducing the risk of catastrophic forgetting. 72.4 Training Infrastructure 

Pre-training PRAGMA on 207 B tokens spanning 24 B user events introduces several engineering challenges. The heterogeneous, table-structured nature of the data requires specialised storage, batching, and truncation strategies. We describe each in turn below. 

Data Storage. The pre-training corpus is stored as a two-level structure: a user index (an LMDB-backed key-value store mapping each user to their tokenised profile state and per-user token statistics) and a collection of event shards 

(Parquet files partitioned by event count, so each file contains only users with the same number of events). This layout allows workers to stream event shards independently and look up profile state on demand. 

Batching. Each training sample consists of a complete event history together with its associated profile state tokens. Because event histories vary greatly in length, from a handful of events to thousands, naïve padding-based batching would waste the majority of compute on padding tokens. Sharding records by event count avoids many random-access disk operations during loading and yields uniform-length event sequences within each batch, so the History Encoder operates on a rectangular tensor without ragged or padded dimensions. We employ dynamic batching with a fixed token budget that fits into GPU memory: records from the same shard are greedily packed until the budget is reached. 

Sequence Packing. Within a batch, individual events still vary in their number of tokens. Rather than padding every event to the longest one, we pack all event tokens into a flat buffer and process them with a variable-length (varlen) attention kernel (Dao et al., 2022), so tokens from different events do not attend to each other at this stage. Together with shard-based batching, this eliminates padding overhead along both the event and token axes. Compared to a padded baseline, sequence packing coupled with dynamic batching yields a 2–5× throughput improvement, depending on the sequence length distribution in the dataset. 

Truncation. To bound memory consumption at a fixed context length, we apply two levels of truncation before packing. At the event level , each individual event is truncated to at most 24 tokens, affecting only 0.01 % of events. At the profile state level , the static profile state sequence is truncated to at most 200 tokens. Users with zero events are discarded; users with more than 6,500 events are subsampled by retaining the most recent ones, preserving temporal recency. 

Pre-training Compute. The three model variants were trained with bf16 mixed precision and the Muon optimiser combined with AdamW (Loshchilov et al., 2019; Jordan, 2024; Liu et al., 2025). PRAGMA-S (10 M parameters) and PRAGMA-M (100 M) were trained on 16 × NVIDIA H100 GPUs, and PRAGMA-L (1 B) on 32 × NVIDIA H100 GPUs. The smallest variant converged in approximately 2 days, while the 100 M and 1 B models each required roughly 2 weeks of wall-clock time. 

## 3 Evaluation 

For commercial sensitivity reasons, we do not report absolute downstream metrics and instead express all results as relative changes with respect to a task-specific reference. Throughout the paper, relative performance is computed as 

(x/ baseline − 1) % , where x is the score of the evaluated method. 

3.1 Evaluation Protocol 

We evaluate PRAGMA primarily via embedding probes and Low-Rank Adaptation (LoRA) (Hu et al., 2022) fine-tuning on downstream tasks. 

3.1.1 Embedding Probing 

Embedding probing facilitates rapid iteration during experimentation before committing to LoRA fine-tuning, e.g., to gauge whether a new feature brings the expected gain, to select a checkpoint after a pre-training run for further evaluation, or to determine whether it is worth exploring a task as a downstream target at all. The embeddings are extracted from the History Encoder output ( zh). 8For our probing analysis, we evaluate the [USR] token, the final [EVT] token, and a combination of both, using a standard linear probe. Given a downstream task with predefined train, validation, and test partitions, we first for-ward each record through the frozen encoder to obtain fixed-size representations and then train a linear probe (logistic or linear regression) on the training partition. We observe that probe performance is robust to the choice of hyper-parameters, so fitting a probe typically takes a couple of minutes. Since our architecture is inherently “pre-norm”, the embeddings were standard-scaled prior to probe fitting. We found that training the probe with the L-BFGS opti-miser (Liu et al., 1989) yields the best results and converges quickly. We note that while Gradient Boosted Decision Trees (GBDT) perform well on lower-dimensional embeddings (e.g., 

192 -d), the requirement for per-task hyper-parameter tuning and the increased time-to-fit make them less practical than linear probing for high-velocity model evaluation. 

3.1.2 Downstream Adaptation with LoRA 

To specialise the PRAGMA backbone for downstream tasks, we employ Low-Rank Adaptation (LoRA), which in-troduces a minimal parameter overhead of only 2–4 %. In this setup, the pre-trained weights are fine-tuned for task-specific objectives to bridge the gap between general representation learning and downstream requirements. We apply LoRA to QKV projections and MLP layers within encoder layers, following a common practice (Hu et al., 2022; Dettmers et al., 2023), and default to rank = 8 with α = 8 across all experiments, but also sweep the rank across 

{4, 8, 16 } on smaller datasets. We use the Adam optimiser (Kingma et al., 2015) for LoRA fine-tuning, and training typically uses 1/8 of the wall-clock time used during pre-training, converging in 12 hours to a few days depending on the dataset size. 

3.1.3 Preparing Downstream Datasets 

For each downstream task, we obtain a unique identifier, which typically consists of a profile id and an evaluation point. Next, we gather the event history and profile attributes directly preceding the evaluation point. We follow the pre-defined folds and splits for each downstream task. The downstream dataset collection process mirrors that of the pre-training dataset. 

3.2 Downstream Tasks Credit Scoring. The task is to assess credit risk for retail applications by predicting the probability of default within the first 12 months of use. The downstream dataset spans multiple years and is diverse across records. This task is cast as a binary classification problem with a minority class, and performance is measured with ROC-AUC and PR-AUC offline metrics. 

Communication Engagement. The task is to predict whether a user who abandoned a credit application mid-process will open a re-engagement communication. This action serves as an upper-funnel proxy for resuming the application and eventually originating a loan. A distinguishing aspect of this task is the severely limited sample size, requiring the model to capture nuanced event-level signals from minimal data. This task is formulated as a binary classification problem, and the main offline metrics are ROC-AUC and PR-AUC. 

External Fraud. This task is a representative fraud detection use case formulated as a binary classification problem. Performance is evaluated using precision and recall as the primary offline metrics. 

Product Recommendation. The task is to predict which products a user is likely to adopt in the near future, con-ditioned on receiving a specific communication (e.g., email or push notification). A key challenge lies in modelling conversion propensity across multiple products simultaneously while accounting for the contextual influence of the communication. The task is formulated as a multilabel classification problem, where the model outputs independent probabilities of conversion for each product in the portfolio. Performance is evaluated using mean average precision (mAP) as the primary offline metric. 

Recurrent Transactions. This task focuses on predicting whether a given transaction corresponds to a recurring subscription that will repeat in the following month. A key challenge lies in distinguishing true recurring patterns 9Task Metric Baseline (ref.) PRAGMA 

Credit scoring PR-AUC – +130.2 % ROC-AUC – +12.4 % Comm. engagement PR-AUC – +79.4 % ROC-AUC – +20.4 % External fraud Precision – +16.7 % Recall – +64.7 % Product rec. mAP – +40.5 % Recurrent txns F1 – +5.8 % Lifetime value PR-AUC – +1.8 % ROC-AUC – +2.6 % Table 2: PRAGMA significantly outperforms internal task-specific models while sharing most of the parameters across tasks. The relative performance is computed as (PRAGMA /baseline − 1). The large variant with LoRA fine-tuning is used as PRAGMA. from irregular or one-off payments given limited historical signals. The problem is formulated as a binary classification task, and performance is evaluated using macro-averaged F1-score to account for class imbalance and ensure balanced performance across classes. 

Lifetime Value (LTV). The LTV task is to assess the probability of a user generating positive gross profit, and is formulated as a binary classification problem. A distinguishing aspect of the LTV dataset is that users have shorter event histories, e.g., a couple of weeks, while the prediction horizon is typically 6 months or more. The main offline metrics are ROC-AUC and PR-AUC. 

3.3 Main Results 

The results presented in Table 2 demonstrate that PRAGMA consistently outperforms existing task-specific baselines across nearly all evaluated domains, despite sharing most of its parameters across tasks. The most striking improve-ments are observed in precision-recall metrics for high-impact tasks: PR-AUC increased by 130.2 % in Credit Scoring and 79.4 % in Communication Engagement, suggesting that PRAGMA is exceptionally effective at identifying low-frequency, high-value signals where traditional models struggle. While ROC-AUC gains are more tempered, they remain substantial at +12.4 % and +20.4 % for the same tasks, respectively. Although performance is more compa-rable on tasks like Lifetime Value and Recurrent Transactions, the overall trend confirms that PRAGMA provides a superior universal representation that matches or exceeds the performance of isolated, task-specific models. 

3.3.1 Effect of Model Scale 

The results in Table 3 illustrate the performance impact of scaling the PRAGMA architecture from the Small (S, 10 M) variant to the Medium (M, 100 M) and Large (L, 1 B) variants. We observe that scaling gains are highly task-dependent, with the most significant improvements concentrated in Credit Scoring, where the Large model achieves a +35.2 % boost in PR-AUC and a +5.8 % gain in ROC-AUC over the Small reference. Notably, the scaling behaviour for Communication Engagement is non-monotonic; the Medium variant exhibits a slight ROC-AUC regression ( −1.8 %), while the Large variant recovers to +0.7 %. For more stable metrics like Recurrent Transactions and LTV, performance gains are more modest, typically remaining under +3.5 %. These results suggest that while increasing parameter count generally enhances predictive power, the Small model already provides a highly competitive representation for transactional and lifetime value predictions, offering a potential efficiency sweet spot for those specific production use cases. 10 PRAGMA Task Metric S (ref.) M L

External fraud Precision – +12.0 % +16.4 % Recall – +24.8 % +23.5 % Product rec. mAP – +18.9 % +27.0 % Credit scoring PR-AUC – +16.3 % +35.2 % ROC-AUC – +3.6 % +5.8 % Lifetime value PR-AUC – +1.5 % +3.0 % ROC-AUC – +1.7 % +3.4 % Comm. engagement PR-AUC – +0.1 % +1.6 % ROC-AUC – −1.8 % +0.7 % Recurrent txns F1 – +0.6 % +0.4 % Table 3: Model performance scales with parameter count . The performance is relative to PRAGMA-S fine-tuned with LoRA and computed as (model /PRAGMA-S − 1). 

PRAGMA-M Task Metric Scratch (ref.) LoRA 

Comm. engagement PR-AUC – +18.6 % ROC-AUC – +5.0 % Credit scoring PR-AUC – +13.0 % ROC-AUC – +1.6 % Product rec. mAP – +10.3 % Recurrent txns F1 – +0.6 % Lifetime value PR-AUC – +0.4 % ROC-AUC – +0.3 % Table 4: Performance comparison of LoRA fine-tuning against task-specific models trained from scratch. Rel-ative performance is computed as (LoRA /Scratch − 1). LoRA consistently matches or exceeds the performance of full-parameter training from scratch. 

3.3.2 Effect of Pre-training 

The results in Table 4 validate our approach, demonstrating that LoRA fine-tuning consistently matches or exceeds the performance of full-parameter training from scratch across all evaluated tasks. The largest gains are observed in Communication Engagement, where LoRA achieves +18.6 % in PR-AUC and +5.0 % in ROC-AUC, suggesting that the pre-trained PRAGMA backbone captures rich diverse event patterns that are difficult to learn when training a model from scratch on a single downstream task. Credit Scoring follows a similar pattern, with LoRA yielding a +13.0 % improvement in PR-AUC and a +1.6 % lift in ROC-AUC. Product Recommendation also benefits substantially, with a +10.3 % gain in mAP. For Recurrent Transactions and Lifetime Value, the improvements are more modest (+0.6 % F1,and +0.4 % / +0.3 % PR-AUC / ROC-AUC respectively), indicating that the scratch-trained baselines already capture most of the task-relevant structure for these objectives, and LoRA fine-tuning maintains parity without regression. These findings are particularly significant for production environments, as they confirm that PRAGMA can consolidate multiple independent, high-maintenance models into a single shared system without sacrificing predictive accuracy, while maintaining a significantly smaller trainable parameter footprint. 11 PRAGMA-S PRAGMA-M PRAGMA-L Task Metric Emb. LoRA Emb. LoRA Emb. LoRA 

Product rec. mAP – +57.2 % – +68.4 % – +68.1 % External fraud Precision – +30.8 % – +29.8 % – +23.8 % Recall – +27.4 % – +24.5 % – +13.3 % Comm. engagement PR-AUC – +72.9 % – +49.7 % – +54.1 % ROC-AUC – +16.9 % – +11.2 % – +13.5 % Credit scoring PR-AUC – +18.0 % – +20.4 % – +10.3 % ROC-AUC – +0.2 % – +2.4 % – +1.5 % Recurrent txns F1 – +4.5 % – +3.2 % – +2.3 % Lifetime value PR-AUC – +3.6 % – +2.4 % – +2.9 % ROC-AUC – +4.7 % – +3.4 % – +3.9 % Table 5: Relative improvement of LoRA-tuned models over embedding-only baselines across scales. For each model size (S, M, L), the embedding-only variant is used as the reference (Emb). Performance gains are computed as (LoRA /Emb − 1). 

3.4 Additional Experiments and Ablations 3.4.1 Effect of Low-Rank Adaptation 

As shown in Table 5, across all evaluated tasks and model scales, the LoRA-tuned variants consistently outperform the embedding-only baselines, demonstrating the efficacy of parameter-efficient fine-tuning in capturing task-specific nuances that fixed embeddings may miss. The most substantial improvements are observed in Communication En-gagement, where LoRA delivers a remarkable +72.9 % gain in PR-AUC for the Small model and maintains significant leads in the Medium and Large variants. In Credit Scoring, we see a peak relative improvement of +20.4 % in PR-AUC for the Medium model, suggesting that LoRA layers are particularly effective at this scale for complex classification. Gains in Recurrent Transactions and LTV are more modest, typically ranging from +2.3 % to +4.7 %. 

3.4.2 Effect of Profile State 

Table 6 isolates the contribution of the Profile State Encoder (§2.3) by comparing the full PRAGMA-S model against a variant that removes the profile-state branch entirely, relying solely on event-level representations. The impact is strongly task-dependent. Credit Scoring benefits substantially, with a +31.8 % relative gain in PR-AUC and +4.9 % in ROC-AUC. The outsized PR-AUC improvement indicates that profile state is particularly valuable for identifying the minority default class, where static signals such as account tenure and onboarding characteristics provide discrim-inative context that event sequences alone cannot fully capture. In contrast, Lifetime Value shows more moderate gains of +2.2 % in PR-AUC and +2.0 % in ROC-AUC, suggesting that gross-profit likelihood is largely inferable from transactional patterns over the prediction horizon. Communication Engagement exhibits a slight PR-AUC regression (−3.0 %) alongside a marginal ROC-AUC gain (+1.3 %), indicating that re-engagement propensity is driven almost entirely by pre-drop-off event patterns rather than static user characteristics. These results validate the two-branch design of PRAGMA: the dedicated Profile State Encoder adds significant value for tasks where static profile state is informative, while the architecture degrades gracefully when those signals are less relevant. 

3.4.3 Communication Engagement (Uplift) 

This task moves beyond conversion prediction to optimal treatment selection: the goal is to identify which messaging strategy best re-engages users with abandoned credit applications. The dataset is smaller in scale than our other downstream benchmarks, yet large-scale pre-training proves decisive, significantly outperforming a baseline trained on the limited in-domain data alone. As an uplift task, it also offers a distinct evaluation angle — PRAGMA is used as a frozen feature extractor feeding a meta-learner rather than being fine-tuned, isolating representational quality in the absence of task-specific adaptation. 12 PRAGMA-S Task Metric Event-only (ref.) Full 

External fraud Precision – +46.8 % Recall – +85.6 % Credit scoring PR-AUC – +31.8 % ROC-AUC – +4.9 % Product rec. mAP – +3.5 % Lifetime value PR-AUC – +2.2 % ROC-AUC – +2.0 % Recurrent txns F1 – +2.4 % Comm. engagement PR-AUC – −3.0 % ROC-AUC – +1.3 % Table 6: Profile state contributes substantially to tasks where static user characteristics are discriminative. The relative performance is computed as (Full /Event-only − 1). 

Task Metric Baseline (ref.) PRAGMA 

Comm. engagement (uplift) AUUC – +163.7 % SNIPS – +10.8 % Table 7: Performance comparison of PRAGMA-L against the internal uplift baseline using the same meta-learner framework. The relative performance is computed as (PRAGMA-L /Baseline − 1). Concretely, we adopt a meta-learner framework (Künzel et al., 2019) to estimate heterogeneous treatment effects, requiring the model to capture complex interactions between pre-drop-off event signals, profile state, and treatment assignment. Both PRAGMA and the baseline use the same meta-learner, differing only in the underlying representa-tion. Table 7 summarises results using Area Under the Uplift Curve (AUUC) and SNIPS (Swaminathan et al., 2015). PRAGMA-L’s ability to capture latent event-level patterns translates to highly effective treatment allocation, achieving a relative AUUC increase of 163.7 % over the internal baseline. 

3.4.4 Effect of a Pre-trained Text Encoder 

In the standard PRAGMA architecture, text values are learned jointly with all other tabular features via an embedding lookup table (see §2.3.1). To prevent the model from underfitting sparse, noisy, or highly irregular financial text (e.g., truncated transaction descriptions), we investigate offloading text comprehension to a dedicated, pre-trained text embedding model, e.g., Nemotron-1B-v2 (de Souza P. Moreira et al., 2024). This decoupled approach provides richer, out-of-the-box semantics and frees the primary Event Transformer (§2.3.3) to focus on cross-feature interactions. While we do not use this as the default formulation in our generalized core architecture, we report on it as an optional extension that offers valuable domain-specific insights. 

Implementation Details. The addition of a pre-trained text encoder involves multiple structural changes to the PRAGMA architecture. First, for semantic types (keys) whose values are normally encoded using a custom-trained BPE tokeniser and a trainable embedding lookup table, we instead use the frozen pre-trained model to map the com-plete text string to a single vector, which is then adapted via a one-layer trainable projection (see Figure 5). Second, instead of reconstructing exact token labels for these text fields during MLM optimisation (see §2.3.5), we train PRAGMA to reconstruct the continuous text embedding produced by the pre-trained text encoder with Mean Squared Error (MSE). 13 Nemotron    

> Emb
> PRAGMA + Nemotron PRAGMA
> Projection
> metal plan
> met plan
> 736 1146
> al
> 2687
> metal plan Description
> 4062
> Emb
> Description
> 4062
> Emb

Figure 5: Text embedding with PRAGMA (left) compared to a version with pre-trained Nemotron-1B-v2 text embedding (right) . Instead of our custom trained BPE tokeniser and a trainable embedding lookup table, a pre-trained “frozen” Nemotron maps an entire text value to a single text embedding vector which is projected into the Transformer’s base dimension with a trainable projection. 

PRAGMA-M Task Metric ref. +Nemotron 

Credit scoring PR-AUC – +16.1 % ROC-AUC – +2.8 % Recurrent txns F1 – +0.1 % Lifetime value PR-AUC – +0.8 % ROC-AUC – +0.6 % External fraud Precision – +3.8 % Recall – −0.7 % Product rec. mAP – −6.4 % Table 8: Impact of pre-trained text embeddings on downstream tasks is concentrated in text-heavy domains. 

The performance is estimated relative to a LoRA-tuned PRAGMA-M. 

Results & Discussion. The results are shown in Table 8. Downstream effects track how much label-relevant signal sits in free text versus categorical and behavioural structure. Credit Scoring shows the clearest upside, with +16.1 % relative PR-AUC and +2.8 % ROC-AUC under Nemotron. Product Recommendation instead loses ground: mAP drops by 6.4 % relative, plausibly because sparse text adds little beyond what the structural channels already encode. External Fraud moves modestly and in opposite directions on precision (+3.8 %) versus recall ( −0.7 %), while LTV and Recurrent Transactions stay near flat on the reported metrics. Because this variant also increases PRAGMA-M training latency by about 18 %, we keep it as an opt-in module for text-heavy tasks rather than baking it into the default architecture. 

3.4.5 Limitations in Highly Relational Tasks: Anti-Money Laundering 

We formulate Anti-Money Laundering (AML) as a binary classification task. As shown in Table 9, this is a setting where PRAGMA significantly underperforms the production baseline. We attribute this performance gap to two primary factors. First, the downstream AML dataset is sufficiently large for the baseline model to learn robust task-specific representations without requiring foundation-level pre-training. Second, and more critically, AML detection is inherently relational: the baseline leverages cross-record features that capture network-level signals. Because PRAGMA processes event histories in isolation, the resulting embeddings do not inherently capture the cross-record dependency structures crucial for this task. 14 Task Metric Baseline (ref.) PRAGMA 

Anti-money laundering F0.5 – −47.1 % Table 9: Performance comparison of PRAGMA against baseline for Anti-Money Laundering. The relative performance is computed as (PRAGMA /Baseline − 1) using linear probe on PRAGMA-L embeddings. Performance is evaluated primarily using F0.5 , as it emphasises precision while still accounting for recall. PRAGMA suffers a 47.1 % drop in F0.5 compared to the network-aware baseline, demonstrating that isolated record-level repre-sentations may be insufficient for this highly relational domain. Addressing this limitation remains a key direction for future work. 

## 4 Related Work 

4.1 Transformer 

The landscape of sequence modelling was fundamentally reshaped by the introduction of the Transformer architec-ture (Vaswani et al., 2017), which dispensed with recurrent layers in favour of a parallelisable self-attention mech-anism. Following this, the field branched out into encoder-only models like BERT (Devlin et al., 2019), optimised for discriminative tasks, and decoder-only architectures like GPT-3 (Brown et al., 2020), which catalysed the cur-rent generative AI era through massive scaling and emergent in-context learning. Subsequent research has extended the architecture’s reach via the Vision Transformer (ViT) (Dosovitskiy et al., 2021) for visual perception and the T5 framework (Raffel et al., 2020) for unified text-to-text processing. Recent advancements have prioritised computa-tional efficiency and multimodality, notably through hardware-aware optimisations like FlashAttention (Dao et al., 2022) and the adoption of Mixture-of-Experts (MoE) (Fedus et al., 2022) in models like Mixtral 8×7B (Jiang et al., 2024). In the current paradigm, models such as Gemini 1.5 (Gemini Team, 2024) and GPT-4o (Hurst et al., 2024) have moved beyond compositional architectures to native multimodality, enabling seamless reasoning across diverse data streams. In this landscape, PRAGMA should be understood as an encoder foundation model for heterogeneous tabular event streams. Although motivated by financial transactions, it extends naturally to any domain where entities accumulate irregular, multi-field records over time. It inherits the scalability and bidirectional contextualisation of encoder-only Transformers, adapting them to heterogeneous fields, explicit time signals, and reusable record-level representations. 

4.2 Masked Modelling 

Parallel to the scaling of generative decoders, masked modelling established a dominant paradigm for self-supervised representation learning. This was pioneered by BERT (Devlin et al., 2019), which utilised a Masked Language Mod-elling (MLM) objective to capture bidirectional context, a technique further refined by RoBERTa (Liu et al., 2019) through dynamic masking and optimised training recipes. The success of MLM was later translated to the vision do-main via Masked Image Modelling (MIM), with BEiT (Bao et al., 2021) and Masked Autoencoders (MAE) (He et al., 2022) demonstrating that reconstructing obscured image patches forces the model to learn holistic structural repre-sentations. Recent trends have moved towards cross-modal unification, as seen in Data2Vec (Baevski et al., 2022), and a shift from raw signal reconstruction to latent feature prediction, exemplified by the Joint-Embedding Predictive Architecture (I-JEPA) (Assran et al., 2023). PRAGMA is directly inspired by this line of work, but extends masked modelling from text and images to hetero-geneous financial records. Our objective masks individual tokens, whole events, and semantic types, encouraging the reconstruction of partially observed events and the learning of transferable representations from full transaction histories. 

4.3 Transformers for Tabular Data 

While Gradient Boosted Decision Trees (GBDTs) have historically dominated structured data, the Transformer has spurred a new class of “Tabular Deep Learning” architectures. Early entries like TabTransformer (Huang et al., 15 2020) and FT-Transformer (Gorishniy et al., 2021) focused on modelling inter-feature dependencies through self-attention, demonstrating performance parity with GBDTs on high-dimensional datasets. This was improved by SAINT (Somepalli et al., 2021), which introduced a dual-attention mechanism for both feature and row interactions, and Trompt (Chen et al., 2023), which proposed prompt-tuning to disentangle intrinsic table properties from sample variations. A paradigm shift occurred with TabPFN (Hollmann et al., 2023), a foundation model pre-trained on syn-thetic data to approximate Bayesian inference. Leveraging in-context learning, TabPFN generates predictions via a single forward pass, eliminating the need for iterative training. While the original model was restricted to 1,000 sam-ples, TabPFN-v2 and TabPFN-v2.5 (Hollmann et al., 2025; Grinsztajn et al., 2025) scaled the architecture to handle 100,000 samples and real-world complexities, providing native support for categorical features, missing values, and outliers. Most recently, Mitra (Zhang et al., 2025) has adopted the dual-attention mechanism of SAINT but follows the foundation model paradigm of TabPFN by being pre-trained exclusively on a massive mixture of synthetic priors. PRAGMA is related in spirit to tabular Transformers because it preserves field identity and models cross-field inter-actions with attention, but unlike TabTransformer, FT-Transformer, and SAINT, it does not operate on a fixed-schema single row. Compared with TabPFN-style tabular foundation models trained on synthetic supervised tasks, PRAGMA is pre-trained with self-supervision on real financial ledgers and models variable-length user histories of heterogeneous events with a hierarchical encoder. 

4.4 Modelling for Recommender Systems 

Sequential recommendation models share structural similarities with transaction modelling, as both process ordered event sequences with rich side information. Transformer-based recommenders treat user interaction histories as token sequences: SASRec (Kang et al., 2018) replaced recurrence with self-attention to capture long-range dependencies, and BERT4Rec (Sun et al., 2019) demonstrated that bidirectional context via masked item prediction yields more robust representations. The field later converged with the LLM paradigm: P5 (Geng et al., 2022) cast diverse rec-ommendation tasks into a unified text-to-text framework built on T5, while TALLRec (Bao et al., 2023) introduced instruction tuning to align general-purpose LLMs with recommendation logic. More recent industrial work has shifted from modelling only positive interactions to encoding richer event streams. Generative Recommenders (Zhai et al., 2024) interleave item and action tokens in a causal sequence, scaling to trillions of parameters with power-law quality gains. ARGUS (Khrylchenko et al., 2025) decomposes autoregressive learning into feedback and next-item prediction, scaling recommender Transformers to one billion parameters. The TransAct line of work (Xia et al., 2023; 2025) embeds each user action as a composite of content, action type, and context for CTR prediction, and extends to lifelong action sequences. PRAGMA is close to this literature in its use of ordered event histories and self-supervised pre-training. Unlike recommendation models that often reduce each interaction to an item token, PRAGMA models richer financial events with typed fields, amounts, free text, and temporal coordinates, and is adapted to a broader set of banking tasks beyond ranking. 

4.5 Foundation Models for Finance 

The paradigm of financial foundation models has rapidly matured from specialised text encoders to comprehensive reasoning engines that integrate diverse data modalities. This evolution began with FinBERT (Yang et al., 2020), which adapted the encoder-only architecture to financial corpora, establishing a rigorous baseline for discriminative tasks like sentiment analysis and ESG classification. The field shifted toward massive generative scale with BloombergGPT (Wu et al., 2023), which demonstrated that interleaving proprietary financial datasets with general web corpora yields superior performance on domain-specific benchmarks. To address the accessibility barriers of such massive models, FinGPT (Yang et al., 2023) introduced a data-centric, lightweight adaptation framework, democratising access to financial LLMs via efficient LoRA fine-tuning (Hu et al., 2022) of open-source models. Most recently, research has transcended textual boundaries to address the structured nature of market data; models like Time-LLM (Jin et al., 2024) and Chronos (Ansari et al., 2024) treat numerical time series as token sequences, enabling Transformers to perform zero-shot forecasting. Extending this structural shift to consumer finance, recent foundation models are now being trained directly on massive-scale user transaction ledgers. For instance, nuFormer (Braithwaite et al., 2025) demonstrates that jointly 16 fusing tokenised transaction sequences with traditional tabular features can effectively replace manual feature en-gineering for real-world risk prediction. Concurrently, TransactionGPT (Dou et al., 2025) introduces a specialised 3D-Transformer architecture to explicitly model the multimodal, temporal, and tabular dimensions of billion-scale payment trajectories, achieving state-of-the-art performance in downstream anomaly detection and trajectory genera-tion. PRAGMA differs from text-centric financial foundation models such as FinBERT, BloombergGPT, and FinGPT, which primarily operate on financial language, and from Time-LLM or Chronos, which tokenise numerical time series for forecasting. It is closer to transaction-ledger models such as nuFormer and TransactionGPT, but aims for a reusable encoder backbone over multi-source banking events with explicit profile state and lightweight adaptation across diverse discriminative tasks. 

## 5 Conclusion 

We presented PRAGMA, a family of encoder-style foundation models for multi-source banking user histories. PRAGMA combines a key–value–time tokenisation scheme with two encoder branches for profile state and events whose outputs are fused by a history encoder, and is pre-trained with masked modelling on large-scale, heterogeneous financial records. Across diverse downstream tasks—credit scoring, fraud detection, communication engagement, product recommendation, recurrent transaction detection, lifetime value prediction, and more—a single pre-trained backbone achieves superior performance directly from raw banking event sequences, providing a general-purpose representation layer for financial applications. Our experiments reveal several practical insights. LoRA fine-tuning consistently matches or exceeds full training from scratch while updating only a small fraction of parameters, confirming that the pre-trained representations transfer effectively across tasks. Scaling from 10 M to 1 B parameters yields large gains on harder tasks such as credit scoring, while smaller models already provide competitive representations for tasks such as lifetime value prediction, offer-ing a practical efficiency trade-off. The dedicated profile state encoder proves particularly valuable for tasks where static contextual attributes are informative, such as credit scoring and fraud detection, while the architecture degrades gracefully when those signals are less relevant. We also find that integrating a pre-trained text encoder improves per-formance in text-dense domains but adds training overhead that is not justified for text-sparse tasks. Finally, the AML case study highlights a clear limitation: tasks that depend on cross-record relational structure remain out of reach for a model that processes event histories in isolation. These results suggest that multi-source banking event sequences admit transferable representations in much the same way as text and vision, despite their heterogeneous structure, irregular timing, and operational constraints. Extend-ing the model to capture cross-record interactions for relational tasks such as anti-money laundering is a promising direction for future work. 

Acknowledgments 

We thank Dmitry Mittov, Ian Iakobsen, Aleksandr Pushin, Muhammad Anas, Viacheslav Karpov, Nathalie Skrzypek, Leyla Sultanova, Francisco Sanz Estevez, Nikita Kravchuk, Tadas Krisciunas, Amey Baokar, Hanna Danilovich, Jyoti Prakash Bal, Vitalii Radchenko, Kade Main, Nic Hatia, and other Revoluters for their contributions to this work. 

## References 

Abdul Fatir Ansari, Lorenzo Stella, Caner Turkmen, Xiyuan Zhang, Pedro Mercado, Huibin Shen, Oleksandr Shchur, Syama Sundar Rangapuram, Sebastian Pineda Arango, Shubham Kapoor, et al. Chronos: Learning the language of time series. Transactions on Machine Learning Research , 2024. Mahmoud Assran, Quentin Duval, Ishan Misra, Piotr Bojanowski, Pascal Vincent, Michael Rabbat, Yann LeCun, and Nicolas Ballas. Self-supervised learning from images with a joint-embedding predictive architecture. In Conference on Computer Vision and Pattern Recognition , 2023. Alexei Baevski, Wei-Ning Hsu, Qiantong Xu, Arun Babu, Jiatao Gu, and Michael Auli. Data2vec: A general frame-work for self-supervised learning in speech, vision and language. In International Conference on Machine Learning ,2022. 17 Hangbo Bao, Li Dong, Songhao Piao, and Furu Wei. BEiT: BERT pre-training of image transformers. arXiv preprint arXiv:2106.08254 , 2021. Keqin Bao, Jizhi Zhang, Yang Zhang, Wenjie Wang, Fuli Feng, and Xiangnan He. TALLRec: An effective and efficient tuning framework to align large language model with recommendation. In ACM Conference on Recommender Systems , 2023. Rishi Bommasani, Drew A. Hudson, Ehsan Adeli, Russ Altman, Simran Arora, Sydney von Arx, Michael S. Bernstein, Jeannette Bohg, Antoine Bosselut, Emma Brunskill, Erik Brynjolfsson, S. Buch, Dallas Card, Rodrigo Castellon, Niladri S. Chatterji, Annie S. Chen, Kathleen A. Creel, Jared Davis, Dora Demszky, Chris Donahue et al. On the opportunities and risks of foundation models. ArXiv , 2021. DT Braithwaite, Misael Cavalcanti, R Austin McEver, Hiroto Udagawa, Daniel Silva, Rohan Ramanath, Felipe Mene-ses, Arissa Yoshida, Evan Wingert, Matheus Ramos, et al. Your spending needs attention: Modeling financial habits with transformers. arXiv preprint arXiv:2507.23267 , 2025. Tom Brown, Benjamin Mann, Nick Ryder, Melanie Subbiah, Jared D Kaplan, Prafulla Dhariwal, Arvind Neelakantan, Pranav Shyam, Girish Sastry, Amanda Askell, et al. Language models are few-shot learners. Advances in Neural Information Processing Systems , 2020. Mathilde Caron, Hugo Touvron, Ishan Misra, Hervé Jégou, Julien Mairal, Piotr Bojanowski, and Armand Joulin. Emerging properties in self-supervised vision transformers. In International Conference on Computer Vision , 2021. Kuan-Yu Chen, Ping-Han Chiang, Hsin-Rung Chou, Ting-Wei Chen, and Darby Tien-Hao Chang. Trompt: Towards a better deep neural network for tabular data. In International Conference on Machine Learning , 2023. Tri Dao, Dan Fu, Stefano Ermon, Atri Rudra, and Christopher Ré. FlashAttention: Fast and memory-efficient exact attention with io-awareness. Advances in Neural Information Processing Systems , 2022. Gabriel de Souza P. Moreira, Radek Osmulski, Mengyao Xu, Ronay Ak, Benedikt Schifferer, and Even Oldridge. Nv-retriever: Improving text embedding models with effective hard-negative mining. arXiv preprint arXiv:2407.15831 ,2024. doi: 10.48550/arXiv.2407.15831. Tim Dettmers, Artidoro Pagnoni, Ari Holtzman, and Luke Zettlemoyer. QLoRA: Efficient finetuning of quantized llms. Advances in Neural Information Processing Systems , 2023. Jacob Devlin, Ming-Wei Chang, Kenton Lee, and Kristina Toutanova. Bert: Pre-training of deep bidirectional trans-formers for language understanding. In North American Association for Computational Linguistics - Human Lan-guage Technologies , 2019. Alexey Dosovitskiy, Lucas Beyer, Alexander Kolesnikov, Dirk Weissenborn, Xiaohua Zhai, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer, Georg Heigold, Sylvain Gelly, Jakob Uszkoreit, and Neil Houlsby. An image is worth 16x16 words: Transformers for image recognition at scale. In International Conference on Learning Representations , 2021. Yingtong Dou, Zhimeng Jiang, Tianyi Zhang, Mingzhi Hu, Zhichao Xu, Shubham Jain, Uday Singh Saini, Xiran Fan, Jiarui Sun, Menghai Pan, et al. Transactiongpt. arXiv preprint arXiv:2511.08939 , 2025. William Fedus, Barret Zoph, and Noam Shazeer. Switch Transformers: Scaling to trillion parameter models with simple and efficient sparsity. Journal of Machine Learning Research , 2022. Gemini Team. Gemini 1.5: Unlocking multimodal understanding across millions of tokens of context. arXiv preprint arXiv:2403.05530 , 2024. Shijie Geng, Shuchang Liu, Zuohui Fu, Yingqiang Ge, and Yongfeng Zhang. Recommendation as language processing (RLP): A unified pretrain, personalized prompt & predict paradigm (P5). In ACM Conference on Recommender Systems , 2022. Yury Gorishniy, Ivan Rubachev, Valentin Khrulkov, and Artem Babenko. Revisiting deep learning models for tabular data. Advances in Neural Information Processing Systems , 2021. 18 Yury Gorishniy, Ivan Rubachev, and Artem Babenko. On embeddings for numerical features in tabular deep learning. 

Advances in Neural Information Processing Systems , 2022. Léo Grinsztajn, Klemens Flöge, Oscar Key, Felix Birkel, Philipp Jund, Brendan Roof, Benjamin Jäger, Dominik Safaric, Simone Alessi, Adrian Hayler, et al. TabPFN-2.5: Advancing the state of the art in tabular foundation models. arXiv preprint arXiv:2511.08667 , 2025. Kaiming He, Xinlei Chen, Saining Xie, Yanghao Li, Piotr Dollár, and Ross Girshick. Masked autoencoders are scalable vision learners. In Computer Vision and Pattern Recognition , 2022. Dan Hendrycks and Kevin Gimpel. Gaussian error linear units (gelus). arXiv preprint arXiv:1606.08415 , 2016. Noah Hollmann, Samuel Müller, Katharina Eggensperger, and Frank Hutter. TabPFN: A transformer that solves small tabular classification problems in a second. In International Conference on Learning Representations , 2023. Noah Hollmann, Samuel Müller, Lennart Purucker, Arjun Krishnakumar, Max Körfer, Shi Bin Hoo, Robin Tibor Schirrmeister, and Frank Hutter. Accurate predictions on small data with a tabular foundation model. Nature , 2025. Edward J Hu, yelong shen, Phillip Wallis, Zeyuan Allen-Zhu, Yuanzhi Li, Shean Wang, Lu Wang, and Weizhu Chen. LoRA: Low-rank adaptation of large language models. In International Conference on Learning Representations ,2022. Xin Huang, Ashish Khetan, Milan Cvitkovic, and Zohar Karnin. TabTransformer: Tabular data modeling using contextual embeddings. arXiv preprint arXiv:2012.06678 , 2020. Aaron Hurst, Adam Lerer, Adam P Goucher, Adam Perelman, Aditya Ramesh, Aidan Clark, AJ Ostrow, Akila Weli-hinda, Alan Hayes, Alec Radford, et al. GPT-4o system card. arXiv preprint arXiv:2410.21276 , 2024. Albert Q Jiang, Alexandre Sablayrolles, Antoine Roux, Arthur Mensch, Blanche Savary, Chris Bamford, Deven-dra Singh Chaplot, Diego de las Casas, Emma Bou Hanna, Florian Bressand, et al. Mixtral of experts. arXiv preprint arXiv:2401.04088 , 2024. Ming Jin, Shiyu Wang, Lintao Ma, Zhixuan Chu, James Y Zhang, Xiaoming Shi, Pin-Yu Chen, Yuxuan Liang, Yuan-Fang Li, Shirui Pan, and Qingsong Wen. Time-LLM: Time series forecasting by reprogramming large language models. In International Conference on Learning Representations , 2024. Keller Jordan. Muon: An optimizer for hidden layers in neural networks, 2024. URL https://kellerjordan. github.io/posts/muon/ .Wang-Cheng Kang and Julian McAuley. Self-attentive sequential recommendation. In International Conference on Data Mining , 2018. Kirill Khrylchenko, Artem Matveev, Sergei Makeev, and Vladimir Baikalov. Scaling recommender transformers to one billion parameters. arXiv preprint arXiv:2507.15994 , 2025. Diederik P Kingma and Jimmy Ba. Adam: A method for stochastic optimization. In International Conference on Learning Representations , 2015. Alexander Kirillov, Eric Mintun, Nikhila Ravi, Hanzi Mao, Chloe Rolland, Laura Gustafson, Tete Xiao, Spencer Whitehead, Alexander C Berg, Wan-Yen Lo, et al. Segment anything. In Computer Vision and Pattern Recognition ,2023. Sören R Künzel, Jasjeet S Sekhon, Peter J Bickel, and Bin Yu. Metalearners for estimating heterogeneous treatment effects using machine learning. Proceedings of the national academy of sciences , 2019. Dong C Liu and Jorge Nocedal. On the limited memory bfgs method for large scale optimization. Mathematical programming , 1989. Jingyuan Liu, Jianlin Su, Xingcheng Yao, Zhejun Jiang, Guokun Lai, Yulun Du, Yidao Qin, Weixin Xu, Enzhe Lu, Junjie Yan, et al. Muon is scalable for LLM training. arXiv preprint arXiv:2502.16982 , 2025. 19 Yinhan Liu, Myle Ott, Naman Goyal, Jingfei Du, Mandar Joshi, Danqi Chen, Omer Levy, Mike Lewis, Luke Zettlemoyer, and Veselin Stoyanov. RoBERTa: A robustly optimized bert pretraining approach. arXiv preprint arXiv:1907.11692 , 2019. Ilya Loshchilov and Frank Hutter. Decoupled weight decay regularization. In International Conference on Learning Representations , 2019. Colin Raffel, Noam Shazeer, Adam Roberts, Katherine Lee, Sharan Narang, Michael Matena, Yanqi Zhou, Wei Li, and Peter J Liu. Exploring the limits of transfer learning with a unified text-to-text transformer. Journal of Machine Learning Research , 2020. Rico Sennrich, Barry Haddow, and Alexandra Birch. Neural machine translation of rare words with subword units. In 

Annual Meeting of the Association for Computational Linguistics , 2016. Gowthami Somepalli, Micah Goldblum, Avi Schwarzschild, C Bayan Bruss, and Tom Goldstein. SAINT: Improved neural networks for tabular data via row attention and contrastive pre-training. arXiv preprint arXiv:2106.01342 ,2021. Nitish Srivastava, Geoffrey Hinton, Alex Krizhevsky, Ilya Sutskever, and Ruslan Salakhutdinov. Dropout: a simple way to prevent neural networks from overfitting. Journal of Machine Learning Research , 2014. Jianlin Su, Murtadha Ahmed, Yu Lu, Shengfeng Pan, Wen Bo, and Yunfeng Liu. RoFormer: Enhanced transformer with rotary position embedding. Neurocomputing , 2024. Fei Sun, Jun Liu, Jian Wu, Changhua Pei, Xiao Lin, Wenwu Ou, and Peng Jiang. BERT4Rec: Sequential recommen-dation with bidirectional encoder representations from transformer. In International Conference on Information and Knowledge Management , 2019. Adith Swaminathan and Thorsten Joachims. The self-normalized estimator for counterfactual learning. In NeurIPS ,2015. Christian Szegedy, Vincent Vanhoucke, Sergey Ioffe, Jon Shlens, and Zbigniew Wojna. Rethinking the inception architecture for computer vision. In Computer Vision and Pattern Recognition , 2016. Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N Gomez, Łukasz Kaiser, and Illia Polosukhin. Attention is all you need. Advances in Neural Information Processing Systems , 2017. Shijie Wu, Ozan Irsoy, Steven Lu, Vadim Dabravolski, Mark Dredze, Sebastian Ghaffari, Binyam Gebre, Abra-ham Ittycheriah, and Gideon Mann. BloombergGPT: A large language model for finance. arXiv preprint arXiv:2303.17564 , 2023. Xue Xia, Pong Eksombatchai, Nikil Pancha, Dhruvil Deven Badani, Po-Wei Wang, Neng Gu, Saurabh Vishwas Joshi, Nazanin Farahpour, Zhiyuan Zhang, and Andrew Zhai. TransAct: Transformer-based realtime user action model for recommendation at Pinterest. In ACM SIGKDD , 2023. Xue Xia, Saurabh Vishwas Joshi, Kousik Rajesh, Kangnan Li, Yangyi Lu, Nikil Pancha, Dhruvil Deven Badani, Jiajing Xu, and Pong Eksombatchai. TransAct V2: Lifelong user action sequence modeling on Pinterest recommendation. 

arXiv preprint arXiv:2506.02267 , 2025. Ruibin Xiong, Yunchang Yang, Di He, Kai Zheng, Shuxin Zheng, Chen Xing, Huishuai Zhang, Yanyan Lan, Liwei Wang, and Tieyan Liu. On layer normalization in the transformer architecture. In International Conference on Machine Learning , 2020. Hongyang Yang, Xiao-Yang Liu, and Christina Dan Wang. FinGPT: Open-source financial large language models. In 

International Joint Conference on Artificial Intelligence (IJCAI) Symposium on Financial Large Language Models ,2023. Yi Yang, Mark Christopher Siy Uy, and Allen Huang. FinBERT: A pretrained language model for financial commu-nications. arXiv preprint arXiv:2006.08097 , 2020. 20 Jiaqi Zhai, Lucy Liao, Xing Liu, Yueming Wang, Rui Li, Xuan Cao, Leon Gao, Zhaojie Gong, Fangda Gu, Michael He, Yinghai Lu, and Yu Shi. Actions speak louder than words: Trillion-parameter sequential transducers for generative recommendations. arXiv preprint arXiv:2402.17152 , 2024. Xiyuan Zhang, Danielle C. Maddix, Junming Yin, Nick Erickson, Abdul Fatir Ansari, Boran Han, Shuai Zhang, Leman Akoglu, Christos Faloutsos, Michael W. Mahoney, Cuixiong Hu, Huzefa Rangwala, George Karypis, and Bernie Wang. Mitra: Mixed synthetic priors for enhancing tabular foundation models. Advances in Neural Information Processing Systems , 2025. 21

# data

> Data loading, tokenization, and dataset assembly for fastpragma

All internal processing uses polars.

In [ ]:
#| default_exp data

In [ ]:
#| export
import polars as pl, numpy as np, pandas as pd, os, json, tempfile
from datetime import datetime, timezone
from dataclasses import dataclass, field
from collections import defaultdict
from fastcore.all import *
from fastai.data.external import untar_data, URLs
from tokenizers import Tokenizer as HFTokenizer, models, pre_tokenizers, trainers, decoders
from fastprogress.fastprogress import progress_bar

In [ ]:
path = untar_data(URLs.ML_100k)
profile_df = pl.scan_csv(path / "u.user", separator="|",  has_header=False, new_columns=["user_id", "age", "gender", "occupation", "zip_code"])
events_df  = pl.scan_csv(path / "u.data", separator="\t", has_header=False, new_columns=["user_id", "movie_id", "rating", "timestamp"]
    ).with_columns(pl.from_epoch("timestamp", time_unit="s").alias("timestamp")).sort("user_id", "timestamp")

## SourceSchema

Declare each data source with its column types. Profile sources omit `time_col` — all fields share the eval-point timestamp. Event sources provide a `time_col` for ordering.

In [ ]:
#| export
#| export
class DataSource:
    def __init__(self, 
                df: pl.LazyFrame, 
                cats: list = None, 
                conts: list = None,
                signed_conts:list=None,
                texts: list = None,
                time_col: str = None,
                lifelong: list = None,
                num_buckets: int = 100,
                cardinality_threshold: int = 100,
                entity_col: str = 'entity_id',
                is_profile: bool = False,
                name: str='file1'):
        schema = df.collect_schema()
        cats,conts,signed_conts,texts,lifelong = [ifnone(o,[]) for o in (cats,conts,signed_conts,texts,lifelong)]
        all_cols = {*cats, *conts, *texts, *lifelong, entity_col, *([time_col] if time_col else [])}
        self.col_names = schema.names()
        assert all_cols.issubset(schema.names())
        assert set(signed_conts).issubset(conts)
        store_attr()
        if is_profile:
            has_dups = df.select(pl.col(entity_col).is_duplicated().any()).collect().item()
            assert not has_dups, f"profile source must be unique by {entity_col}"


    def __repr__(self):
        return (
            f"DataSource(columns={self.col_names}, name={self.name} "
            f"cats={self.cats or '[]'}, conts={self.conts or '[]'}, "
            f"texts={self.texts or '[]'}, time_col={self.time_col!r})\n"
            f"{self.df.head().collect()}"
        )
    
    @classmethod
    def from_df(cls, df, **kwargs): return cls(pl.from_pandas(df).lazy(), **kwargs)

    @classmethod
    def from_file(cls, path, **kwargs):
        path = Path(path)
        scans = dict(csv=pl.scan_csv, tsv=pl.scan_csv, parquet=pl.scan_parquet, ipc=pl.scan_ipc, feather=pl.scan_ipc)
        scan_kw_names = {'separator','has_header','new_columns','schema','schema_overrides','null_values','try_parse_dates'}
        scan_kw = {k:v for k,v in kwargs.items() if k in scan_kw_names}
        src_kw = {k:v for k,v in kwargs.items() if k not in scan_kw_names}
        suffix = path.suffix[1:].lower()
        scan = scans.get(suffix, pl.scan_csv)
        if suffix == 'tsv': scan_kw = dict(separator='\t', **scan_kw)
        return cls(scan(path, **scan_kw), **src_kw)

In [ ]:
# Test 1: Explicit column types
df = pl.LazyFrame({'cat_col': ['a', 'b', 'a'], 'num_col': [1.0, 2.0, 3.0], 'id_col': [10, 20, 30]})
s = DataSource(df, cats=['cat_col'], conts=['num_col'], entity_col='id_col')
test_eq(sorted(s.cats), ['cat_col'])
test_eq(s.conts, ['num_col'])

# Test 2: Explicit overrides take priority
s2 = DataSource(df, cats=['cat_col'], conts=['num_col', 'id_col'], entity_col='id_col')
test_eq(sorted(s2.cats), ['cat_col'])
test_eq(sorted(s2.conts), ['id_col', 'num_col'])

# Test 3: time_col is excluded from cats/conts
df3 = pl.LazyFrame({'id_col': [1,2], 'evt': ['x', 'y'], 'val': [1, 2], 'ts': [100, 200]})
s3 = DataSource(df3, cats=['evt'], conts=['val'], time_col='ts', entity_col='id_col')
test_eq('ts' not in s3.cats + s3.conts, True)

# Test 4: Float32
df4 = pl.LazyFrame({'f32': [1.0, 2.0]}, schema={'f32': pl.Float32})
s4 = DataSource(df4, conts=['f32'], entity_col='f32')
test_eq(s4.conts, ['f32'])

In [ ]:
profile = DataSource.from_file(path / "u.user", separator="|", has_header=False, new_columns=["user_id","age","gender","occupation","zip_code"], 
                            cats=["gender","occupation"], conts=["age"], entity_col="user_id")
profile

DataSource(columns=['user_id', 'age', 'gender', 'occupation', 'zip_code'], name=file1 cats=['gender', 'occupation'], conts=['age'], texts=[], time_col=None)
shape: (5, 5)
┌─────────┬─────┬────────┬────────────┬──────────┐
│ user_id ┆ age ┆ gender ┆ occupation ┆ zip_code │
│ ---     ┆ --- ┆ ---    ┆ ---        ┆ ---      │
│ i64     ┆ i64 ┆ str    ┆ str        ┆ str      │
╞═════════╪═════╪════════╪════════════╪══════════╡
│ 1       ┆ 24  ┆ M      ┆ technician ┆ 85711    │
│ 2       ┆ 53  ┆ F      ┆ other      ┆ 94043    │
│ 3       ┆ 23  ┆ M      ┆ writer     ┆ 32067    │
│ 4       ┆ 24  ┆ M      ┆ technician ┆ 43537    │
│ 5       ┆ 33  ┆ F      ┆ other      ┆ 15213    │
└─────────┴─────┴────────┴────────────┴──────────┘

In [ ]:
pdf = pd.DataFrame({'cat': ['x','y','x'], 'val': [1,2,3], 'id': [10,20,30]})
s5 = DataSource.from_df(pdf, cats=['cat'], conts=['val'], entity_col='id')
test_eq(sorted(s5.cats), ['cat'])
test_eq(s5.conts, ['val'])
test_eq(s5.entity_col, 'id')
test_eq(s5.time_col, None)

## Key-Value-Time Conversion

Convert wide DataFrames to the uniform `(key, value, value_type, time)` format used internally. Profile rows get `time=0` (all at eval point); events use their timestamp column; lifelong columns carry their own datetime.

In [ ]:
#| export
class Tokenizer:
    def __init__(self, num_buckets=100, cardinality_threshold=100):
        store_attr()
        self.key_vocab = {'[PAD]': 0, '[MASK]': 1, '[UNK]': 2, '[USR]': 3, '[EVT]': 4}
        self.val_vocab = {'[PAD]': 0, '[MASK]': 1, '[UNK]': 2}
        self.num_bounds = {}
        self.bpe_tokenizer = None
        self.bpe_offset = 0
        self.lowcard_text_keys = set()
    def _add_key(self, k): self.key_vocab.setdefault(k, len(self.key_vocab))
    def _add_val(self, v): self.val_vocab.setdefault(v, len(self.val_vocab))

lets parse time first only lifelong columns can be datetime? for event its the specified datetime column?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, that's right:

- **Regular profile fields** → no time at all (time=0, all at eval point)
- **Lifelong columns** → each has its own datetime (e.g. `first_topup` at 2020-11-02 12:09:04)
- **Events** → use the `time_col` from the `DataSource` (e.g. the `timestamp` column in the events `DataSource`)

Your `_parse_time` already handles all three cases cleanly — `None` → 0, `datetime` → epoch int, and plain `int` passes through.

how to make this efficient

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The key is avoiding row-by-row iteration. With polars you can do it all lazily using columnar operations:

- **`.melt()` / `.unpivot()`** to go wide → long in one shot. Keep a pre-built mapping of column → value type (`cat`/`cont`/`text`) so you don't branch per-row
- **Separate melts** for profile, events, and lifelong — they have different time semantics so merging them early creates mess
- **`.with_columns(pl.lit(0))`** for profile's time column instead of computing per-row
- **Do the datetime → epoch conversion as a single `.cast(pl.Int64)` or `pl.col(t).dt.epoch()`**, not with Python's `datetime.timestamp()`

The lazy query engine will fuse the operations and only materialize when you `.collect()` at the end.

In [ ]:
@patch
def _melt_profile_fields(self:Tokenizer, src:DataSource):
    melt_cols = src.cats + src.conts + src.texts
    if not melt_cols: return pl.LazyFrame(schema=[src.entity_col, 'key', 'value', 'vtype', 'time'])
    return (
        src.df
        .select(src.entity_col, *melt_cols)
        .unpivot(index=src.entity_col, on=melt_cols, variable_name='key', value_name='value')
        .with_columns(
            pl.col('value').cast(pl.Utf8),
            pl.when(pl.col('key').is_in(src.cats)).then(pl.lit('cat'))
             .when(pl.col('key').is_in(src.conts)).then(pl.lit('num'))
             .otherwise(pl.lit('text')).alias('vtype'),
            pl.lit(0).cast(pl.Int64).alias('time')
        )
    )

In [ ]:
df = pl.LazyFrame(dict(uid=[1,2], gender=['M','F'], age=[25,30]))
src = DataSource(df, cats=['gender'], conts=['age'], entity_col='uid')
tok = Tokenizer()
res = tok._melt_profile_fields(src).collect()

test_eq(res.shape, (4,5))
test_eq(set(res.columns), {'uid','key','value','vtype','time'})
test_eq(res.filter(pl.col('key')=='gender')['vtype'].unique().to_list(), ['cat'])
test_eq(sorted(res.filter(pl.col('key')=='gender')['value'].to_list()), ['F','M'])
test_eq(res.filter(pl.col('key')=='age')['vtype'].unique().to_list(), ['num'])
test_eq(res['time'].unique().to_list(), [0])
test_eq('uid' in res['key'].to_list(), False)

src = DataSource(pl.LazyFrame(dict(uid=[1])), entity_col='uid')
test_eq(tok._melt_profile_fields(src).collect().height, 0)

In [ ]:
@patch
def _melt_events(self:Tokenizer, src:DataSource):
    if not src.time_col: raise ValueError(f"{src.name}: events require time_col")
    melt_cols = src.cats + src.conts + src.texts
    if not melt_cols: return pl.LazyFrame(schema=[src.entity_col, 'key', 'value', 'vtype', 'time', 'event_idx'])

    df = (
        src.df
        .with_row_index('_row_id')
        .sort(src.entity_col, src.time_col, '_row_id')
        .with_columns(pl.int_range(0, pl.len()).over(src.entity_col).alias('event_idx'))
    )

    time_col_type = df.collect_schema()[src.time_col]
    time_expr = (
        pl.col(src.time_col).dt.epoch(time_unit='s').cast(pl.Int64)
        if time_col_type.is_temporal()
        else pl.col(src.time_col).cast(pl.Int64)
    )

    return (
        df
        .select(src.entity_col, src.time_col, 'event_idx', *melt_cols)
        .unpivot(index=[src.entity_col, src.time_col, 'event_idx'], on=melt_cols, variable_name='key', value_name='value')
        .with_columns(
            pl.col('value').cast(pl.Utf8),
            pl.when(pl.col('key').is_in(src.cats)).then(pl.lit('cat'))
             .when(pl.col('key').is_in(src.conts)).then(pl.lit('num'))
             .otherwise(pl.lit('text')).alias('vtype'),
            time_expr.alias('time')
        )
        .drop(src.time_col)
        .sort(src.entity_col, 'event_idx')
    )

In [ ]:
df = pl.LazyFrame(dict(uid=[1,1,2], evt=['click','view','click'], amt=[10,20,30], ts=[100,200,300]))
src = DataSource(df, cats=['evt'], conts=['amt'], time_col='ts', entity_col='uid')
tok = Tokenizer()
res = tok._melt_events(src).collect()

test_eq(res.shape, (6,6))
test_eq(set(res.columns), {'uid','key','value','vtype','time','event_idx'})
test_eq(sorted(res.filter(pl.col('uid')==1)['event_idx'].unique().to_list()), [0,1])
test_eq(res.filter(pl.col('uid')==2)['event_idx'].unique().to_list(), [0])
test_eq(sorted(res['time'].unique().to_list()), [100,200,300])
test_eq(res.filter(pl.col('key')=='evt')['vtype'].unique().to_list(), ['cat'])
test_eq(res.filter(pl.col('key')=='amt')['vtype'].unique().to_list(), ['num'])

In [ ]:
df = pl.LazyFrame(dict(
    uid=[1,1],
    evt=['login','logout'],
    amt=[5,10],
    ts=[datetime(2020,6,15,12,0,0,tzinfo=timezone.utc), datetime(2020,6,15,12,30,0,tzinfo=timezone.utc)]))
src = DataSource(df, cats=['evt'], conts=['amt'], time_col='ts', entity_col='uid')
tok = Tokenizer()
res = tok._melt_events(src).collect()
exp = pl.Series(df.select('ts').collect()['ts']).dt.epoch(time_unit='s').to_list()

test_eq(sorted(res['time'].unique().to_list()), exp)
test_eq(len(res['time'].unique()), 2)

In [ ]:
@patch
def _melt_lifelong(self:Tokenizer, src:DataSource):
    if not src.lifelong: return pl.LazyFrame(schema=[src.entity_col, 'key', 'value', 'vtype', 'time'])

    parts = []
    for lc in src.lifelong:
        parts.append(
            src.df
            .select(src.entity_col, lc)
            .filter(pl.col(lc).is_not_null())
            .with_columns(
                pl.lit('lifelong').alias('key'),
                pl.lit(lc).alias('value'),
                pl.lit('cat').alias('vtype'),
                pl.col(lc).dt.epoch(time_unit='s').cast(pl.Int64).alias('time'),
            )
            .drop(lc)
        )

    return pl.concat(parts)

In [ ]:
df = pl.LazyFrame(dict(
    uid=[1,2],
    first_topup=[datetime(2020,1,1), datetime(2021,6,15)],
    first_trade=[datetime(2020,3,10), None],
    extra=[99,100]))
src = DataSource(df, conts=['extra'], lifelong=['first_topup','first_trade'], entity_col='uid')
tok = Tokenizer()
res = tok._melt_lifelong(src).collect()

test_eq(res.shape, (3,5))
test_eq(set(res.columns), {'uid','key','value','vtype','time'})
test_eq(res['key'].unique().to_list(), ['lifelong'])
test_eq(res['vtype'].unique().to_list(), ['cat'])
test_eq(set(res['value'].unique().to_list()), {'first_topup','first_trade'})
test_eq(res.filter(pl.col('uid')==1, pl.col('value')=='first_topup')['time'].item(), df.select(pl.col('first_topup').first().dt.epoch(time_unit='s')).collect().item())

src = DataSource(df, lifelong=[], entity_col='uid')
test_eq(tok._melt_lifelong(src).collect().height, 0)

In [ ]:
@patch
def _split_signed_nums(self:Tokenizer, kvt, signed_nums):
    signed_nums = list(signed_nums)
    signed = kvt.filter(pl.col('key').is_in(signed_nums), pl.col('vtype') == 'num').with_columns(pl.col('value').cast(pl.Float64, strict=False).alias('_fval'))
    mag = signed.with_columns(pl.col('_fval').abs().cast(pl.Utf8).alias('value')).drop('_fval')
    sign = signed.with_columns(
        (pl.col('key') + '_sign').alias('key'),
        pl.when(pl.col('_fval').is_null()).then(pl.lit('unknown')).when(pl.col('_fval') < 0).then(pl.lit('negative')).when(pl.col('_fval') > 0).then(pl.lit('positive')).otherwise(pl.lit('zero')).alias('value'),
        pl.lit('cat').alias('vtype')
    ).drop('_fval')
    rest = kvt.filter(~(pl.col('key').is_in(signed_nums) & (pl.col('vtype') == 'num')))
    return pl.concat([rest, mag, sign], how='diagonal_relaxed')

In [ ]:
@patch
def _to_kvt(self:Tokenizer, src:DataSource):
    regular = self._melt_profile_fields(src) if src.is_profile else self._melt_events(src)
    lifelong = self._melt_lifelong(src)
    parts = [p for p in [regular, lifelong] if len(p.collect_schema()) > 0]
    if not parts: return pl.LazyFrame(schema=[src.entity_col, 'key', 'value', 'vtype', 'time'])
    res = pl.concat(parts, how='diagonal_relaxed')
    res = self._split_signed_nums(res, src.signed_conts)
    res = res.with_columns(pl.col('value').cast(pl.Utf8).fill_null(''), pl.col('time').fill_null(0))
    keep = [c for c in [src.entity_col, 'key', 'value', 'vtype', 'time', 'event_idx'] if c in res.collect_schema().names()]
    return res.select(keep)

In [ ]:
df = pl.LazyFrame(dict(
    uid=list(range(1,4)),
    gender=['M','F','I'],
    age=[25,30,-30],
    first_topup=[datetime(2020,1,1), datetime(2021,6,15), datetime(2021,6,15)],
    first_trade=[datetime(2020,3,10), None, datetime(2021,6,15)]))
src = DataSource(df, cats=['gender'], conts=['age'], signed_conts=['age'], lifelong=['first_topup','first_trade'], entity_col='uid', is_profile=True)
tok = Tokenizer()
res = tok._to_kvt(src).collect()

test_eq(res.shape, (14,5))
test_eq(set(res.columns), {'uid','key','value','vtype','time'})
test_eq(set(res.filter(pl.col('key')=='age_sign')['value'].to_list()), {'positive','negative'})
test_eq(res.filter(pl.col('key')=='age')['value'].cast(pl.Float64).abs().to_list(), [25.,30.,30.])
test_eq(res.filter(pl.col('value').is_in(['first_topup','first_trade']))['time'].min() > 0, True)
test_eq(res.filter(pl.col('uid')==2, pl.col('value')=='first_trade').height, 0)

In [ ]:
result

key,value,vtype,time,key_id
str,str,str,i64,i64
"""gender""","""M""","""cat""",0,5
"""age""","""25""","""num""",0,6
"""unknown""","""x""","""text""",0,2


## Tokenizer

Builds key and value vocabularies, bucketizes numerical values, and tokenizes source schemas into fixed-size tensors for the model.

- Each key gets one token in the key vocab
- Categorical values: one token per unique value
- Numerical values: percentile-bucket index
- Textual values: BPE subword tokens

Keys use a single shared vocabulary across all sources. `replace_strict` maps known keys to their index; any unknown key gets the `[UNK]` fallback.

In [ ]:
@patch
def _map_keys(self:Tokenizer, kvt):
    return kvt.with_columns(pl.col('key').replace_strict(self.key_vocab, default=self.key_vocab['[UNK]']).alias('key_id'))

In [ ]:
# Test: known keys → correct IDs, unknown key → [UNK]
tok = Tokenizer()
tok._add_key('gender'); tok._add_key('age')

kvt = pl.LazyFrame({
    'key': ['gender', 'age', 'unknown'],
    'value': ['M', '25', 'x'],
    'vtype': ['cat', 'num', 'text'],
    'time': [0, 0, 0]
})
result = tok._map_keys(kvt).collect()
test_eq(result['key_id'].to_list(), [tok.key_vocab['gender'], tok.key_vocab['age'], tok.key_vocab['[UNK]']])

In [ ]:
tok.key_vocab.items(), result

(dict_items([('[PAD]', 0), ('[MASK]', 1), ('[UNK]', 2), ('[USR]', 3), ('[EVT]', 4), ('gender', 5), ('age', 6)]),
 shape: (3, 5)
 ┌─────────┬───────┬───────┬──────┬────────┐
 │ key     ┆ value ┆ vtype ┆ time ┆ key_id │
 │ ---     ┆ ---   ┆ ---   ┆ ---  ┆ ---    │
 │ str     ┆ str   ┆ str   ┆ i64  ┆ i64    │
 ╞═════════╪═══════╪═══════╪══════╪════════╡
 │ gender  ┆ M     ┆ cat   ┆ 0    ┆ 5      │
 │ age     ┆ 25    ┆ num   ┆ 0    ┆ 6      │
 │ unknown ┆ x     ┆ text  ┆ 0    ┆ 2      │
 └─────────┴───────┴───────┴──────┴────────┘)

Categorical values map one-to-one to vocabulary tokens. PRAGMA uses this for high-frequency fields like MCC codes — one value, one token, direct lookup.

In [ ]:
@patch
def _map_cat_vals(self:Tokenizer, kvt):
    return (
        kvt.filter(pl.col('vtype') == 'cat')
        .with_columns(
            pl.col('value').replace_strict(self.val_vocab, default=self.val_vocab['[UNK]']).alias('val_id'),
            pl.lit(0).cast(pl.Int64).alias('val_pos')
        )
    )

In [ ]:
tok = Tokenizer()
tok._add_val('M'); tok._add_val('F')

kvt = pl.LazyFrame({
    'key': ['gender', 'gender', 'gender', 'age'],
    'value': ['M', 'unknown', 'F', '25'],
    'vtype': ['cat', 'cat', 'cat', 'num'],
    'time': [0, 0, 0, 1]
})
result = tok._map_cat_vals(kvt).collect().sort('value')
test_eq(result['val_id'].to_list() ,[ tok.val_vocab['F'], tok.val_vocab['M'], tok.val_vocab['[UNK]'],])

In [ ]:
tok.val_vocab.items(), result

(dict_items([('[PAD]', 0), ('[MASK]', 1), ('[UNK]', 2), ('M', 3), ('F', 4)]),
 shape: (3, 6)
 ┌────────┬─────────┬───────┬──────┬────────┬─────────┐
 │ key    ┆ value   ┆ vtype ┆ time ┆ val_id ┆ val_pos │
 │ ---    ┆ ---     ┆ ---   ┆ ---  ┆ ---    ┆ ---     │
 │ str    ┆ str     ┆ str   ┆ i64  ┆ i64    ┆ i64     │
 ╞════════╪═════════╪═══════╪══════╪════════╪═════════╡
 │ gender ┆ F       ┆ cat   ┆ 0    ┆ 4      ┆ 0       │
 │ gender ┆ M       ┆ cat   ┆ 0    ┆ 3      ┆ 0       │
 │ gender ┆ unknown ┆ cat   ┆ 0    ┆ 2      ┆ 0       │
 └────────┴─────────┴───────┴──────┴────────┴─────────┘)

Continuous values are discretized into percentile bins learned per key during `fit()`.

- `0` maps to the special numerical bucket `<bucket_0>`
- null / missing / unparsable values map to `[UNK]`
- non-zero numeric values map to percentile buckets `<bucket_1>`, `<bucket_2>`, ...
- `np.searchsorted` finds the bucket index from the precomputed boundaries

This keeps all model inputs discrete while preserving approximate magnitude/order information.

In [ ]:
@patch
def _map_num_vals(self:Tokenizer, kvt):
    nums = kvt.filter(pl.col('vtype') == 'num')
    unk = lambda df: df.with_columns(pl.lit(self.val_vocab['[UNK]']).alias('val_id'), pl.lit(0).cast(pl.Int64).alias('val_pos'))
    if not self.num_bounds: return unk(nums)

    def enc_key(k,bounds):
        return (
            nums.filter(pl.col('key') == k)
            .with_columns(pl.col('value').cast(pl.Float64, strict=False).alias('_fval'))
            .with_columns(
                pl.when(pl.col('_fval').is_null()).then(None)
                .when(pl.col('_fval') == 0).then(0)
                .otherwise(pl.col('_fval').map_batches(lambda s: np.searchsorted(bounds, s.to_numpy())+1, return_dtype=pl.Int64)).alias('_bucket_idx'))
            .with_columns(
                pl.when(pl.col('_bucket_idx').is_null()).then(pl.lit(self.val_vocab['[UNK]']))
                .otherwise(pl.format('<bucket_{}>', '_bucket_idx').replace_strict(self.val_vocab, default=self.val_vocab['[UNK]'])).alias('val_id'),
                pl.lit(0).cast(pl.Int64).alias('val_pos'))
            .drop('_fval', '_bucket_idx')
        )

    parts = [enc_key(k,bounds) for k,bounds in self.num_bounds.items()]
    parts.append(unk(nums.filter(~pl.col('key').is_in(list(self.num_bounds)))))
    return pl.concat(parts, how='diagonal_relaxed')

In [ ]:
tok = Tokenizer(num_buckets=4)
for i in range(tok.num_buckets+1): tok._add_val(f'<bucket_{i}>')
tok.num_bounds['amt'] = np.array([2.5, 5.0, 7.5])

kvt = pl.LazyFrame(dict(
    key=['amt','amt','amt','amt','amt','amt','val'],
    value=[None,'0.0','2.5','5.0','-3.0','999.0','9'],
    vtype=['num']*7,
    time=[0]*7))

result = tok._map_num_vals(kvt).collect()

want = {
    None: '[UNK]',
    '0.0': '<bucket_0>',
    '2.5': '<bucket_1>',
    '5.0': '<bucket_2>',
    '-3.0': '<bucket_1>',
    '999.0': '<bucket_4>',
}

for v,b in want.items(): test_eq(result.filter(pl.col('key')=='amt', pl.col('value').eq_missing(v))['val_id'].item(), tok.val_vocab[b])
test_eq(result.filter(pl.col('key')=='val')['val_id'].item(), tok.val_vocab['[UNK]'])

In [ ]:
result

key,value,vtype,time,val_id,val_pos
str,str,str,i64,i64,i64
"""amt""",null,"""num""",0,2,0
"""amt""","""0.0""","""num""",0,3,0
"""amt""","""2.5""","""num""",0,4,0
"""amt""","""5.0""","""num""",0,5,0
"""amt""","""-3.0""","""num""",0,4,0
"""amt""","""999.0""","""num""",0,7,0
"""val""","""9""","""num""",0,2,0


High-cardinality text gets BPE subword tokenization via tiktoken. Each subword becomes its own row sharing the same key — the model groups them later via shared key embeddings and within-field positional encodings.

Using a dynamic `bpe_offset = len(val_vocab)` computed at encode time means BPE IDs naturally shift as `val_vocab` grows — add a new categorical token, and all subword IDs move up by one. That's fine as long as you re-encode the entire dataset whenever you refit. Unknown keys/values/new numeric fields all route to `[UNK]` gracefully, so a new source won't error — it'll just need refitting (and re-encoding) to learn from it properly.

In [ ]:
@patch
def _encode_text(self:Tokenizer, v):
    ids = self.bpe_tokenizer.encode(v).ids
    if not ids: return [self.val_vocab['[UNK]']]
    return [self.bpe_offset+i for i in ids]

In [ ]:
@patch
def _norm_text(self:Tokenizer, v): return '[UNK]' if v is None or v == '' else v

In [ ]:
@patch
def _map_text_vals(self:Tokenizer, kvt):
    lowcard_keys = getattr(self, 'lowcard_text_keys', set())
    txt = kvt.filter(pl.col('vtype') == 'text').with_columns(pl.col('value').map_elements(self._norm_text, return_dtype=pl.Utf8).alias('value'))
    low = txt.filter(pl.col('key').is_in(lowcard_keys)).with_columns(pl.col('value').replace_strict(self.val_vocab, default=self.val_vocab['[UNK]']).alias('val_id'), pl.lit(0).cast(pl.Int64).alias('val_pos'))
    high = txt.filter(~pl.col('key').is_in(lowcard_keys))
    if self.bpe_tokenizer is None and high.limit(1).collect().height: raise ValueError("high-cardinality text present but no BPE tokenizer was fitted")
    if self.bpe_tokenizer is None: return low
    high = high.with_columns(pl.col('value').map_elements(lambda v: self._encode_text(v), return_dtype=pl.List(pl.Int64)).alias('val_ids')).with_columns(pl.int_ranges(0, pl.col('val_ids').list.len()).alias('val_pos')).explode('val_ids', 'val_pos', empty_as_null=True).rename({'val_ids': 'val_id'})
    return pl.concat([low, high], how='diagonal_relaxed')

In [ ]:
tok = Tokenizer()
tok._add_key('desc'); tok._add_key('desc_short'); tok._add_val('hello')
tok.lowcard_text_keys.add('desc_short')

class MockEncoded:
    def __init__(self, ids): self.ids = ids

class MockTok:
    def encode(self, text): return MockEncoded([7,8,7] if 'plan' in text else [9])

tok.bpe_tokenizer = MockTok()
off = 0

kvt = pl.LazyFrame(dict(
    key=['desc', 'desc', 'desc_short'],
    value=['metal plan', 'world', 'hello'],
    vtype=['text']*3,
    time=[0]*3))

res = tok._map_text_vals(kvt).collect()
mp = res.filter(pl.col('key')=='desc', pl.col('value')=='metal plan')
w = res.filter(pl.col('key')=='desc', pl.col('value')=='world')
ds = res.filter(pl.col('key')=='desc_short')

test_eq(mp['val_id'].to_list(), [off+7, off+8, off+7])
test_eq(mp['val_pos'].to_list(), [0,1,2])
test_eq(w['val_id'].item(), off+9)
test_eq(w['val_pos'].item(), 0)
test_eq(ds['val_id'].item(), tok.val_vocab['hello'])
test_eq(ds['val_pos'].item(), 0)
test_eq(res.height, 5)

In [ ]:
result

key,value,vtype,time,val_id,val_pos
str,str,str,i64,i64,i64
"""amt""",null,"""num""",0,2,0
"""amt""","""0.0""","""num""",0,3,0
"""amt""","""2.5""","""num""",0,4,0
"""amt""","""5.0""","""num""",0,5,0
"""amt""","""-3.0""","""num""",0,4,0
"""amt""","""999.0""","""num""",0,7,0
"""val""","""9""","""num""",0,2,0


A single-pass scan builds all vocabularies and parameters: key vocab across all sources, val vocab from categoricals, percentile boundaries for continuous columns, and identifies low-cardinality text fields to treat as categorical.

In [ ]:
@patch
def fit(self:Tokenizer, sources:L, eval_points:pl.LazyFrame=None):
    sources = L(sources)
    kvt = pl.concat(sources.map(self._to_kvt), how='diagonal_relaxed').collect()

    for k in sorted(kvt['key'].unique()): self._add_key(k)

    cat_vals = kvt.filter(pl.col('vtype') == 'cat')['value'].unique()
    for v in sorted(cat_vals): self._add_val(v)

    num_df = kvt.filter(pl.col('vtype') == 'num')
    qs = np.linspace(0, 100, self.num_buckets+1)[1:-1]
    for k in sorted(num_df['key'].unique()):
        vals = num_df.filter(pl.col('key') == k)['value'].cast(pl.Float64, strict=False).drop_nulls()
        vals = vals.filter(vals != 0)
        if len(vals): self.num_bounds[k] = np.percentile(vals.to_numpy(), qs)

    for i in range(self.num_buckets+1): self._add_val(f'<bucket_{i}>')

    text_df = kvt.filter(pl.col('vtype') == 'text')
    high_texts = []
    for k in sorted(text_df['key'].unique()):
        vals = text_df.filter(pl.col('key') == k)['value']
        if vals.n_unique() <= self.cardinality_threshold:
            self.lowcard_text_keys.add(k)
            for v in sorted(vals.unique()): self._add_val(v)
        else: high_texts += vals.to_list()

    self.bpe_offset = len(self.val_vocab)
    if high_texts:
        self.bpe_tokenizer = HFTokenizer(models.BPE(unk_token='[UNK]'))
        self.bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        trainer = trainers.BpeTrainer(vocab_size=28000, special_tokens=['[UNK]', '[PAD]', '[MASK]'], min_frequency=2)
        self.bpe_tokenizer.train_from_iterator(high_texts, trainer)
    else: self.bpe_tokenizer = None

    return self

In [ ]:
#| test
tok = Tokenizer(num_buckets=5)
df = pl.LazyFrame(dict(uid=[1,2], gender=['M','F'], age=[25,30], first_topup=[datetime(2020,1,1), datetime(2021,6,15)]))
src = DataSource(df, cats=['gender'], conts=['age'], lifelong=['first_topup'], entity_col='uid', is_profile=True)
tok.fit(L([src]))

test_eq('gender' in tok.key_vocab, True)
test_eq('age' in tok.key_vocab, True)
test_eq('lifelong' in tok.key_vocab, True)
test_eq('first_topup' in tok.val_vocab, True)
test_eq('M' in tok.val_vocab, True)
test_eq('F' in tok.val_vocab, True)
test_eq('age' in tok.num_bounds, True)
test_eq(len(tok.num_bounds['age']), tok.num_buckets-1)
test_eq([f'<bucket_{i}>' in tok.val_vocab for i in range(tok.num_buckets+1)], [True]*(tok.num_buckets+1))

kvt = tok._to_kvt(src)
res = tok._map_keys(kvt).collect()
test_eq(set(res.columns), {'uid','key','value','vtype','time','key_id'})
test_eq(res.filter(pl.col('key')=='gender')['key_id'].unique().item(), tok.key_vocab['gender'])
test_eq(res.filter(pl.col('key')=='age')['key_id'].unique().item(), tok.key_vocab['age'])

res = tok._map_cat_vals(kvt).collect()
test_eq(set(res['value']), {'M','F','first_topup'})
test_eq((res['val_pos'] == 0).all(), True)

res = tok._map_num_vals(kvt).collect()
test_eq(res.height, 2)
test_eq((res['val_id'] != tok.val_vocab['[UNK]']).all(), True)
test_eq((res['val_pos'] == 0).all(), True)

tok = Tokenizer(num_buckets=5, cardinality_threshold=2)
df = pl.LazyFrame(dict(uid=[1,2,3], high_text=['the quick brown fox','hello world again','something else'], low_text=['x','y','x']))
src = DataSource(df, texts=['high_text','low_text'], entity_col='uid', is_profile=True)
tok.fit(L([src]))

test_eq('low_text' in tok.lowcard_text_keys, True)
test_eq('x' in tok.val_vocab and 'y' in tok.val_vocab, True)
test_eq('high_text' in tok.lowcard_text_keys, False)
test_eq(tok.bpe_tokenizer is not None, True)

txt = tok._map_text_vals(tok._to_kvt(src)).collect()
test_eq(txt.height >= 5, True)
test_eq((txt['val_id'] != tok.val_vocab['[UNK]']).all(), True)

df1 = pl.LazyFrame(dict(uid=[1], cat1=['a'], num1=[10]))
df2 = pl.LazyFrame(dict(uid=[2], cat2=['b'], num2=[20]))
src1 = DataSource(df1, cats=['cat1'], conts=['num1'], entity_col='uid', is_profile=True)
src2 = DataSource(df2, cats=['cat2'], conts=['num2'], entity_col='uid', is_profile=True)
tok = Tokenizer()
tok.fit(L([src1, src2]))

test_eq(all(o in tok.key_vocab for o in ['cat1','cat2','num1','num2']), True)
test_eq(all(o in tok.val_vocab for o in ['a','b']), True)
test_eq(all(o in tok.num_bounds for o in ['num1','num2']), True)

n_keys,n_vals,n_bounds = len(tok.key_vocab),len(tok.val_vocab),len(tok.num_bounds)
tok.fit(L([src1, src2]))
test_eq((len(tok.key_vocab),len(tok.val_vocab),len(tok.num_bounds)), (n_keys,n_vals,n_bounds))

tok = Tokenizer()
src = DataSource(pl.LazyFrame(dict(uid=[1])), entity_col='uid', is_profile=True)
tok.fit(L([src]))
test_eq(tok.key_vocab, {'[PAD]':0, '[MASK]':1, '[UNK]':2, '[USR]':3, '[EVT]':4})
test_eq(tok.num_bounds, {})
test_eq(tok.bpe_tokenizer, None)

## Time Features

Tokenization keeps event timestamps as raw epoch seconds until this stage. Events use log-seconds to the latest event in the entity history; lifelong profile fields use log-seconds to the evaluation point; regular profile fields stay at zero.

In [ ]:
def _to_epoch(ts):
    if isinstance(ts, (int, float)): return int(ts)
    return pl.Series([str(ts)]).str.to_datetime().dt.epoch(time_unit='s').item()

In [ ]:
def _log_s(ts): return 8*np.log1p(np.maximum(ts, 0)/8) # ts in seconds expected

In [ ]:
@patch
def _map_values(self:Tokenizer, kvt):
    cats  = self._map_cat_vals(kvt)
    nums  = self._map_num_vals(kvt)
    texts = self._map_text_vals(kvt)
    return pl.concat([cats, nums, texts], how='diagonal_relaxed')

In [ ]:
@patch
def encode_source(self:Tokenizer, src:DataSource):
    kvt = self._map_keys(self._to_kvt(src))
    vals = self._map_values(kvt)
    keep = [c for c in [src.entity_col,'key','key_id','value','val_id','val_pos','vtype','time','event_idx'] if c in vals.collect_schema().names()]
    return vals.select(keep)

Truncation (max events per user, max tokens per event) belongs in the **dataloader**, not in tokenization. Tokenization produces the full flat parquet per source; the dataloader applies truncation at load time so it can be tuned per experiment without re-tokenizing.

In [ ]:
@patch
def _tokenize(self:Tokenizer, src:DataSource, eval_time=None, entity_max_ts=None):
    if eval_time is None: raise ValueError('eval_time must be provided')
    kvt = self.encode_source(src)
    names = kvt.collect_schema().names()

    if 'event_idx' in names:
        kvt = kvt.join(entity_max_ts, on=src.entity_col, how='left') if entity_max_ts is not None else kvt.with_columns(pl.col('time').max().over(src.entity_col).alias('max_epoch'))
        return kvt.with_columns(
            pl.col('max_epoch').sub(pl.col('time')).map_batches(lambda x: pl.Series(_log_s(x.to_numpy().astype(np.float64))), return_dtype=pl.Float64).alias('logsec'),
            pl.from_epoch('time', time_unit='s').dt.hour().cast(pl.Float32).alias('hour'),
            pl.from_epoch('time', time_unit='s').dt.weekday().cast(pl.Float32).alias('dow'),
            pl.from_epoch('time', time_unit='s').dt.day().cast(pl.Float32).alias('dom')
        ).drop('max_epoch')

    eval_epoch = _to_epoch(eval_time)
    return kvt.with_columns(
        pl.when(pl.col('time') > 0)
        .then(pl.col('time').map_batches(lambda x: pl.Series(_log_s((eval_epoch - x.to_numpy()).astype(np.float64))), return_dtype=pl.Float64))
        .otherwise(0.0).alias('logsec')
    )

In [ ]:
tok = Tokenizer(num_buckets=4)

pdf = pl.LazyFrame(dict(uid=[1,2], plan=['free','pro'], first_seen=[datetime(2020,1,1,tzinfo=timezone.utc), None]))
edf = pl.LazyFrame(dict(uid=[1,1,2], evt=['view','buy','view'], amt=[0,10,20], ts=[100,200,300]))

psrc = DataSource(pdf, cats=['plan'], lifelong=['first_seen'], entity_col='uid', is_profile=True)
esrc = DataSource(edf, cats=['evt'], conts=['amt'], time_col='ts', entity_col='uid')

tok.fit(L([psrc, esrc]))

prof = tok._tokenize(psrc, eval_time=datetime(2020,1,11,tzinfo=timezone.utc)).collect()
evts = tok._tokenize(esrc, eval_time=0).collect()

test_eq(set(['uid','key','key_id','value','val_id','val_pos','vtype','time','logsec']).issubset(prof.columns), True)
test_eq(set(['uid','key','key_id','value','val_id','val_pos','vtype','time','event_idx','logsec','hour','dow','dom']).issubset(evts.columns), True)

test_eq(prof.filter(pl.col('key')=='plan')['logsec'].to_list(), [0.0,0.0])
test_eq(prof.filter(pl.col('value')=='first_seen').height, 1)
test_eq(abs(prof.filter(pl.col('value')=='first_seen')['logsec'].item() - _log_s(10*24*60*60)) < 1e-6, True)

test_eq(evts.filter(pl.col('uid')==1, pl.col('time')==200)['logsec'].unique().item(), 0.0)
test_eq(abs(evts.filter(pl.col('uid')==1, pl.col('time')==100, pl.col('key')=='evt')['logsec'].item() - _log_s(100)) < 1e-6, True)
test_eq(evts.filter(pl.col('uid')==2)['logsec'].unique().to_list(), [0.0])
test_eq(evts.filter(pl.col('uid')==1)['event_idx'].unique().sort().to_list(), [0,1])

test_eq((prof['val_id'] != tok.val_vocab['[UNK]']).all(), True)
test_eq((evts.filter(pl.col('key')!='amt')['val_id'] != tok.val_vocab['[UNK]']).all(), True)

In [ ]:
profile = DataSource.from_file(path / "u.user", separator="|", has_header=False,
    new_columns=["user_id","age","gender","occupation","zip_code"], 
    cats=["gender","occupation"], conts=["age"], entity_col="user_id",
    is_profile=True)
events = DataSource.from_file(path / "u.data", separator="\t", has_header=False,
    new_columns=["user_id", "movie_id", "rating", "timestamp"],
    cats=["movie_id"], conts=["rating"], time_col="timestamp", entity_col="user_id")

In [ ]:
tok = Tokenizer(num_buckets=10, cardinality_threshold=100)
tok.fit(L([profile, events]))

profile_tok = tok._tokenize(profile, eval_time='1998-04-01T00:00:00').head(3).collect()
events_tok = tok._tokenize(events, eval_time='1998-04-01T00:00:00').head(2).collect()

print("Profile:", profile_tok)
print("Events:", events_tok)

Profile: shape: (3, 9)
┌─────────┬────────┬────────┬───────┬───┬─────────┬───────┬──────┬────────┐
│ user_id ┆ key    ┆ key_id ┆ value ┆ … ┆ val_pos ┆ vtype ┆ time ┆ logsec │
│ ---     ┆ ---    ┆ ---    ┆ ---   ┆   ┆ ---     ┆ ---   ┆ ---  ┆ ---    │
│ i64     ┆ str    ┆ i64    ┆ str   ┆   ┆ i64     ┆ str   ┆ i64  ┆ f64    │
╞═════════╪════════╪════════╪═══════╪═══╪═════════╪═══════╪══════╪════════╡
│ 1       ┆ gender ┆ 6      ┆ M     ┆ … ┆ 0       ┆ cat   ┆ 0    ┆ 0.0    │
│ 2       ┆ gender ┆ 6      ┆ F     ┆ … ┆ 0       ┆ cat   ┆ 0    ┆ 0.0    │
│ 3       ┆ gender ┆ 6      ┆ M     ┆ … ┆ 0       ┆ cat   ┆ 0    ┆ 0.0    │
└─────────┴────────┴────────┴───────┴───┴─────────┴───────┴──────┴────────┘


Events: shape: (2, 13)
┌─────────┬──────────┬────────┬───────┬───┬────────────┬──────┬─────┬──────┐
│ user_id ┆ key      ┆ key_id ┆ value ┆ … ┆ logsec     ┆ hour ┆ dow ┆ dom  │
│ ---     ┆ ---      ┆ ---    ┆ ---   ┆   ┆ ---        ┆ ---  ┆ --- ┆ ---  │
│ i64     ┆ str      ┆ i64    ┆ str   ┆   ┆ f64        ┆ f32  ┆ f32 ┆ f32  │
╞═════════╪══════════╪════════╪═══════╪═══╪════════════╪══════╪═════╪══════╡
│ 1       ┆ movie_id ┆ 7      ┆ 168   ┆ … ┆ 115.438142 ┆ 21.0 ┆ 1.0 ┆ 22.0 │
│ 1       ┆ movie_id ┆ 7      ┆ 172   ┆ … ┆ 115.438142 ┆ 21.0 ┆ 1.0 ┆ 22.0 │
└─────────┴──────────┴────────┴───────┴───┴────────────┴──────┴─────┴──────┘


## Tokenizer Persistence

The fitted tokenizer is part of the dataset contract. Save key/value vocabularies, numeric bucket boundaries, low-cardinality text choices, and any BPE model so tokenized train/valid/test data can be reproduced exactly.

In [ ]:
@patch
def save(self:Tokenizer, path='tokenizer.json'):
    path = Path(path)
    state = dict(key_vocab=dict(self.key_vocab), val_vocab=dict(self.val_vocab), num_bounds={k:v.tolist() for k,v in self.num_bounds.items()},
        lowcard_text_keys=list(self.lowcard_text_keys), num_buckets=self.num_buckets, cardinality_threshold=self.cardinality_threshold)
    if self.bpe_tokenizer is not None:
        bpe_path = path.with_name(f'{path.stem}_bpe{path.suffix}')
        self.bpe_tokenizer.save(str(bpe_path))
        state['bpe_path'] = bpe_path.name
    path.write_text(json.dumps(state, indent=2))
    return path

In [ ]:
@patch(cls_method=True)
def load(cls:Tokenizer, path='tokenizer.json'):
    path = Path(path)
    state = json.loads(path.read_text())
    tok = cls(num_buckets=state['num_buckets'], cardinality_threshold=state['cardinality_threshold'])
    tok.key_vocab = state['key_vocab']
    tok.val_vocab = state['val_vocab']
    tok.num_bounds = {k:np.array(v) for k,v in state['num_bounds'].items()}
    tok.lowcard_text_keys = set(state['lowcard_text_keys'])
    tok.bpe_offset = len(tok.val_vocab)
    tok.bpe_tokenizer = HFTokenizer.from_file(str(path.with_name(state['bpe_path']))) if 'bpe_path' in state else None
    return tok

In [ ]:
tmp = Path(tempfile.mkdtemp())/'tokenizer.json'
tok.save(tmp)
tok2 = Tokenizer.load(tmp)
test_eq(tok2.key_vocab, tok.key_vocab)
test_eq(tok2.val_vocab, tok.val_vocab)
test_eq(set(tok2.num_bounds), set(tok.num_bounds))
test_eq(tok2.bpe_offset, len(tok2.val_vocab))

## PRAGMADataset

Assemble profile + event sources into per-entity tokenized sequences. Fits a tokenizer across all sources and produces tokenized tensors keyed by entity.

Usage:
    ds = PRAGMADataset(static=profile_schema, dynamic=[events_schema], entity_col='user_id')
    batches = ds.tokenize()
    ds.show_batch(1)

## Shared Event Time Reference

When multiple event sources exist, each entity needs one global latest timestamp across all event sources. Otherwise every source would incorrectly treat its own latest event as `logsec=0`.

In [ ]:
def _compute_entity_max_ts(sources):
    "Return a LazyFrame with entity_col and max_epoch across all event sources."
    parts,entity_col = [],None
    for src in sources:
        if src.is_profile or not src.time_col: continue
        if entity_col is None: entity_col = src.entity_col
        assert src.entity_col == entity_col
        typ = src.df.collect_schema()[src.time_col]
        time = pl.col(src.time_col).dt.epoch(time_unit='s').cast(pl.Int64) if typ.is_temporal() else pl.col(src.time_col).cast(pl.Int64)
        parts.append(src.df.select(src.entity_col, src.time_col).with_columns(time.alias('epoch')).drop(src.time_col))
    if not parts: return None
    return pl.concat(parts).group_by(entity_col).agg(pl.col('epoch').max().alias('max_epoch'))

In [ ]:
df1 = pl.LazyFrame(dict(uid=[1,1,2], ts=[100,300,200]))
df2 = pl.LazyFrame(dict(uid=[1,2], ts=[500,50]))
prof = DataSource(pl.LazyFrame(dict(uid=[1])), entity_col='uid', is_profile=True)
src1,src2 = [DataSource(o, time_col='ts', entity_col='uid') for o in (df1,df2)]

res = _compute_entity_max_ts([src1]).collect().sort('uid')
test_eq(res['uid'].to_list(), [1,2])
test_eq(res['max_epoch'].to_list(), [300,200])
test_eq(_compute_entity_max_ts([prof]), None)

res = _compute_entity_max_ts([prof, src1, src2]).collect().sort('uid')
test_eq(res['uid'].to_list(), [1,2])
test_eq(res['max_epoch'].to_list(), [500,200])

## Dataset Orchestration

`PRAGMADataset` owns source consistency, tokenizer fitting, and sharded parquet writing. It preserves the caller's `entity_col` name and keeps truncation out of tokenization so dataloader experiments can change context limits without re-tokenizing.

In [ ]:
class PRAGMADataset:
    def __init__(self, profile=None, events=None, entity_col='entity_id', out_path=None):
        store_attr('profile,entity_col')
        self.events,self.tokenizer = L(events or []),None
        self.out_path = Path(ifnone(out_path, './data'))
        os.makedirs(self.out_path, exist_ok=True)
        self._check_sources()

    def _check_sources(self):
        srcs = ([self.profile] if self.profile is not None else []) + list(self.events)
        for i,s in enumerate(srcs):
            if s.entity_col != self.entity_col: raise ValueError(f"source {i} entity_col={s.entity_col!r} != {self.entity_col!r}")

    @property
    def sources(self): return L(([self.profile] if self.profile is not None else []) + list(self.events))

    def fit_tokenizer(self, num_buckets=100, cardinality_threshold=100):
        self.tokenizer = Tokenizer(num_buckets=num_buckets, cardinality_threshold=cardinality_threshold).fit(self.sources)
        tok = self.tokenizer
        print(f"Keys: {len(tok.key_vocab)}, Vals: {len(tok.val_vocab)}, BPE: {tok.bpe_tokenizer.get_vocab_size() if tok.bpe_tokenizer else 'none'}")
        tok.save(self.out_path/'tokenizer.json')
        return tok

Sharded this way because light on memory

In [ ]:
@patch
def tokenize(self:PRAGMADataset, eval_time, n_shards=100):
    "Tokenize sources to entity-sharded parquet files."
    if self.tokenizer is None: self.fit_tokenizer()
    entity_max_ts = _compute_entity_max_ts(self.events)
    parts = []

    if self.profile is not None:
        print('tokenizing profile')
        parts.append(self.tokenizer._tokenize(self.profile, eval_time=eval_time).with_columns(pl.lit(-1).alias('source_idx'), pl.lit(-1).alias('event_idx')))

    for i,src in enumerate(progress_bar(self.events)):
        print(f'tokenizing event source {i}: {src.name}')
        parts.append(self.tokenizer._tokenize(src, eval_time=eval_time, entity_max_ts=entity_max_ts).with_columns(pl.lit(i).alias('source_idx')))

    print('combining sources')
    lf = pl.concat(parts, how='diagonal_relaxed').with_columns(
        pl.col(self.entity_col).hash().mod(n_shards).alias('shard_id'),
        pl.col('hour').cast(pl.Float32).fill_null(0),
        pl.col('dow').cast(pl.Float32).fill_null(0),
        pl.col('dom').cast(pl.Float32).fill_null(0)
    ).sort(self.entity_col, 'source_idx', 'event_idx', 'key', 'val_pos')

    for s in progress_bar(range(n_shards)):
        lf.filter(pl.col('shard_id') == s).sink_parquet(self.out_path/f'shard_{s}.parquet')

    return self.out_path

In [ ]:
tmp = Path(tempfile.mkdtemp())
ds = PRAGMADataset(profile=psrc, events=[esrc], entity_col='uid', out_path=tmp)
tok = ds.fit_tokenizer(num_buckets=4)

test_eq(tok is ds.tokenizer, True)
test_eq((tmp/'tokenizer.json').exists(), True)
test_eq('plan' in tok.key_vocab, True)
test_eq('evt' in tok.key_vocab, True)
test_eq('amt' in tok.key_vocab, True)

Keys: 9, Vals: 13, BPE: none


In [ ]:
out = ds.tokenize(eval_time=datetime(2020,1,11,tzinfo=timezone.utc), n_shards=2)

test_eq(out, tmp)
test_eq(sorted(o.name for o in tmp.glob('*.parquet')), ['shard_0.parquet', 'shard_1.parquet'])

tokenizing profile


 |----------------------------------------| 0.00% [0/1 00:00<?]

tokenizing event source 0: file1


 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

combining sources


 |----------------------------------------| 0.00% [0/2 00:00<?]

 |████████████████████--------------------| 50.00% [1/2 00:00<00:00]

 |████████████████████████████████████████| 100.00% [2/2 00:00<00:00]

In [ ]:
res = pl.concat([pl.scan_parquet(o) for o in tmp.glob('shard_*.parquet')]).collect()
prof = res.filter(pl.col('source_idx') == -1)
evts = res.filter(pl.col('source_idx') == 0)

test_eq(set(['uid','key','key_id','value','val_id','val_pos','vtype','time','logsec','source_idx','event_idx','shard_id']).issubset(prof.columns), True)
test_eq(set(['uid','key','key_id','value','val_id','val_pos','vtype','time','event_idx','logsec','hour','dow','dom','source_idx','shard_id']).issubset(evts.columns), True)

test_eq(prof['source_idx'].unique().to_list(), [-1])
test_eq(prof['event_idx'].unique().to_list(), [-1])
test_eq(evts['source_idx'].unique().to_list(), [0])
test_eq(evts.filter(pl.col('uid')==1, pl.col('time')==200)['logsec'].unique().item(), _log_s(0))
test_eq(abs(evts.filter(pl.col('uid')==1, pl.col('time')==100, pl.col('key')=='evt')['logsec'].item() - _log_s(100)) < 1e-6, True)

In [ ]:
bad_src = DataSource(pl.LazyFrame(dict(user_id=[1], evt=['x'], ts=[1])), cats=['evt'], time_col='ts', entity_col='user_id')
test_fail(lambda: PRAGMADataset(profile=psrc, events=[bad_src], entity_col='uid'))

## End-to-End Tokenization Check

MovieLens 100k is used as a compact integration test: fit the shared tokenizer, write sharded parquet, then verify row counts, token ids, profile/event conventions, and time features.

In [ ]:
out = Path(tempfile.mkdtemp())/'ml100k_tok'
ds = PRAGMADataset(profile=profile, events=[events], entity_col='user_id', out_path=out)
tok = ds.fit_tokenizer(num_buckets=10, cardinality_threshold=100)
ds.tokenize(eval_time='1998-04-01T00:00:00', n_shards=4)
sorted(o.name for o in out.iterdir())

Keys: 10, Vals: 1719, BPE: none


tokenizing profile


 |----------------------------------------| 0.00% [0/1 00:00<?]

tokenizing event source 0: file1


 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

...

 |██████████------------------------------| 25.00% [1/4 00:00<00:02]

 |████████████████████--------------------| 50.00% [2/4 00:01<00:01]

 |██████████████████████████████----------| 75.00% [3/4 00:02<00:00]

 |████████████████████████████████████████| 100.00% [4/4 00:02<00:00]

['shard_0.parquet',
 'shard_1.parquet',
 'shard_2.parquet',
 'shard_3.parquet',
 'tokenizer.json']

In [ ]:
len(tok.key_vocab), len(tok.val_vocab), tok.num_bounds.keys(), tok.bpe_tokenizer

(10, 1719, dict_keys(['age', 'rating']), None)

In [ ]:
toks = pl.concat([pl.scan_parquet(out/f'shard_{i}.parquet') for i in range(4)]).collect()
prof = toks.filter(pl.col('source_idx') == -1)
evts = toks.filter(pl.col('source_idx') == 0)
prof.shape, evts.shape, prof.columns, evts.columns

((2829, 15),
 (200000, 15),
 ['user_id',
  'key',
  'key_id',
  'value',
  'val_id',
  'val_pos',
  'vtype',
  'time',
  'logsec',
  'source_idx',
  'event_idx',
  'hour',
  'dow',
  'dom',
  'shard_id'],
 ['user_id',
  'key',
  'key_id',
  'value',
  'val_id',
  'val_pos',
  'vtype',
  'time',
  'logsec',
  'source_idx',
  'event_idx',
  'hour',
  'dow',
  'dom',
  'shard_id'])

In [ ]:
test_eq(prof.height, 943*3)
test_eq(evts.height, 100000*2)
test_eq(prof['source_idx'].unique().to_list(), [-1])
test_eq(prof['event_idx'].unique().to_list(), [-1])
test_eq(evts['source_idx'].unique().to_list(), [0])
test_eq(set(prof['key'].unique().to_list()), {'gender','occupation','age'})
test_eq(set(evts['key'].unique().to_list()), {'movie_id','rating'})
test_eq((prof['val_id'] != tok.val_vocab['[UNK]']).all(), True)
test_eq((evts['val_id'] != tok.val_vocab['[UNK]']).all(), True)

In [ ]:
u = 1
prof.filter(pl.col('user_id') == u).sort('key')

user_id,key,key_id,value,val_id,val_pos,vtype,time,logsec,source_idx,event_idx,hour,dow,dom,shard_id
i64,str,i64,str,i64,i64,str,i64,f64,i32,i64,f32,f32,f32,u64
1,"""age""",5,"""24""",1711,0,"""num""",0,0.0,-1,-1,0.0,0.0,0.0,0
1,"""gender""",6,"""M""",1686,0,"""cat""",0,0.0,-1,-1,0.0,0.0,0.0,0
1,"""occupation""",8,"""technician""",1706,0,"""cat""",0,0.0,-1,-1,0.0,0.0,0.0,0


In [ ]:
evts.filter(pl.col('user_id') == u).sort('event_idx','key','val_pos').head(12)

user_id,key,key_id,value,val_id,val_pos,vtype,time,logsec,source_idx,event_idx,hour,dow,dom,shard_id
i64,str,i64,str,i64,i64,str,i64,f64,i32,i64,f32,f32,f32,u64
1,"""movie_id""",7,"""168""",759,0,"""cat""",874965478,115.438142,0,0,21.0,1.0,22.0,0
1,"""rating""",9,"""5""",1716,0,"""num""",874965478,115.438142,0,0,21.0,1.0,22.0,0
1,"""movie_id""",7,"""172""",767,0,"""cat""",874965478,115.438142,0,1,21.0,1.0,22.0,0
1,"""rating""",9,"""5""",1716,0,"""num""",874965478,115.438142,0,1,21.0,1.0,22.0,0
1,"""movie_id""",7,"""165""",726,0,"""cat""",874965518,115.438121,0,2,21.0,1.0,22.0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,"""rating""",9,"""4""",1713,0,"""num""",874965556,115.4381,0,3,21.0,1.0,22.0,0
1,"""movie_id""",7,"""196""",793,0,"""cat""",874965677,115.438035,0,4,22.0,1.0,22.0,0
1,"""rating""",9,"""5""",1716,0,"""num""",874965677,115.438035,0,4,22.0,1.0,22.0,0


In [ ]:
chk = evts.filter(pl.col('user_id') == 1, pl.col('key') == 'movie_id').sort('time')
chk.select('user_id','event_idx','time','logsec','hour','dow','dom').head(), chk.select('user_id','event_idx','time','logsec','hour','dow','dom').tail()

(shape: (5, 7)
 ┌─────────┬───────────┬───────────┬────────────┬──────┬─────┬──────┐
 │ user_id ┆ event_idx ┆ time      ┆ logsec     ┆ hour ┆ dow ┆ dom  │
 │ ---     ┆ ---       ┆ ---       ┆ ---        ┆ ---  ┆ --- ┆ ---  │
 │ i64     ┆ i64       ┆ i64       ┆ f64        ┆ f32  ┆ f32 ┆ f32  │
 ╞═════════╪═══════════╪═══════════╪════════════╪══════╪═════╪══════╡
 │ 1       ┆ 0         ┆ 874965478 ┆ 115.438142 ┆ 21.0 ┆ 1.0 ┆ 22.0 │
 │ 1       ┆ 1         ┆ 874965478 ┆ 115.438142 ┆ 21.0 ┆ 1.0 ┆ 22.0 │
 │ 1       ┆ 2         ┆ 874965518 ┆ 115.438121 ┆ 21.0 ┆ 1.0 ┆ 22.0 │
 │ 1       ┆ 3         ┆ 874965556 ┆ 115.4381   ┆ 21.0 ┆ 1.0 ┆ 22.0 │
 │ 1       ┆ 4         ┆ 874965677 ┆ 115.438035 ┆ 22.0 ┆ 1.0 ┆ 22.0 │
 └─────────┴───────────┴───────────┴────────────┴──────┴─────┴──────┘,
 shape: (5, 7)
 ┌─────────┬───────────┬───────────┬───────────┬──────┬─────┬──────┐
 │ user_id ┆ event_idx ┆ time      ┆ logsec    ┆ hour ┆ dow ┆ dom  │
 │ ---     ┆ ---       ┆ ---       ┆ ---       ┆ ---  ┆ --- ┆

In [ ]:
chk = evts.filter(pl.col('user_id') == 1, pl.col('key') == 'movie_id')
test_eq(chk.filter(pl.col('time') == pl.col('time').max())['logsec'].unique().item(), 0.0)
test_eq((chk['logsec'] >= 0).all(), True)
test_eq(chk.sort('time')['logsec'].to_list()[0] >= chk.sort('time')['logsec'].to_list()[-1], True)

can you check these?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

These checks look good overall. The important invariants are passing:

- Tokenized files were written correctly:
  - 4 shard parquet files
  - `tokenizer.json`

- Tokenizer looks reasonable for ML100k:
  - `10` keys
  - `1719` value tokens
  - numeric bounds for `age` and `rating`
  - no BPE needed, since there are no high-cardinality text fields

- Shape checks make sense:
  - profile: `943 users × 3 fields = 2829`
  - events: `100000 ratings × 2 fields = 200000`

- Source conventions look correct:
  - profile rows have `source_idx=-1`, `event_idx=-1`
  - event rows have `source_idx=0`
  - profile keys are `gender`, `occupation`, `age`
  - event keys are `movie_id`, `rating`

- No `[UNK]` values in this toy dataset, which is what we'd hope after fitting/tokenizing the same sources

- Time encoding looks right:
  - user's latest event has `logsec=0`
  - earlier events have positive `logsec`
  - `logsec` decreases as event time approaches the latest event
  - calendar features are populated for event rows and zero-filled for profile rows

One small thing to note: identical timestamps produce multiple `event_idx`s with the same `time` and same `logsec`, e.g. user 1 has multiple ratings at `874965478`. That's fine — `_melt_events` uses `_row_id` as a tie-breaker, so event order is deterministic even when timestamps tie.

I think this is a solid checkpoint. The next natural piece would be a small inspection method like `show_entity` or `show_batch` to make these parquet outputs easy to review without manually filtering.

so we have everything to prepare the dataset as per the paper

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — for the **data preparation / tokenization layer**, we now have the main pieces aligned with the paper:

- source schemas via `DataSource`
- profile/event/lifelong conversion to uniform KVT rows
- categorical values as single value tokens
- numerical values as percentile bucket tokens, including zero bucket
- text values as either low-cardinality categorical tokens or BPE subwords
- shared key vocab
- shared value vocab
- event time as log-seconds to latest event
- event calendar features: hour, day-of-week, day-of-month
- profile regular fields with `logsec=0`
- lifelong profile fields with log-seconds to eval point
- shared tokenizer fit across all sources
- sharded parquet output with profile and event rows

What we **do not yet have** is the later dataloader/model-prep layer:

- group tokenized rows into per-entity profile sequences
- group event rows into per-event token sequences
- insert `[USR]` and `[EVT]` tokens
- construct tensors: `key_id`, `val_id`, `val_pos`, `logsec`, calendar features
- apply max tokens per event / max profile tokens / max events per entity
- batch or pack variable-length event tokens
- optionally build MLM masks

So yes: dataset preparation is in good shape. Next stage is turning the flat tokenized parquet into model-ready samples.

summarize things in this notebook that are related todata that are not done as of now but are kept for later

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Things related to data that we've intentionally **kept for later**:

- **Truncation**
  - Not done during tokenization.
  - Kept for the dataloader so we can change experiment limits without re-tokenizing.
  - Later knobs:
    - max events per entity
    - max tokens per event
    - max profile tokens

- **Grouping flat rows into samples**
  - Current output is flat parquet rows.
  - Later we need to group by entity:
    - profile rows → one profile token sequence
    - event rows → per-event token sequences
    - event sequences → per-user history

- **Adding special tokens**
  - Tokenizer vocab already contains `[USR]` and `[EVT]`.
  - But we haven't inserted them into actual sequences yet.
  - Later:
    - prepend `[USR]` to profile sequence
    - prepend `[EVT]` to each event sequence

- **Tensor assembly**
  - We currently have parquet columns like:

```python
user_id, key_id, val_id, val_pos, logsec, hour, dow, dom, event_idx, source_idx
```

  - Later dataloader should turn these into tensors for the model.

- **Padding / packing**
  - Not implemented yet.
  - Paper uses sequence packing for variable-length event tokens.
  - Later we decide whether to:
    - pad first for simplicity
    - implement packed variable-length attention later

- **Batching**
  - Current tokenizer writes entity-sharded parquet.
  - Later dataloader should create batches, ideally with dynamic token budgets.

- **MLM masking**
  - Not implemented yet.
  - Paper uses:
    - token-level masking
    - event-level masking
    - key-level masking
    - some `[UNK]` replacement
  - This belongs after sample assembly, not in raw tokenization.

- **Train/valid/test splitting**
  - Not handled yet.
  - Tokenizer can fit on train and then encode all splits, but split logic itself still needs to be added.

- **Evaluation point handling beyond a fixed value**
  - We currently pass one `eval_time`.
  - Later real datasets may have per-entity or per-row eval points.
  - Lifelong deltas should then use the correct eval point per entity.

- **More scalable fitting**
  - Current `fit()` collects KVT into memory.
  - Fine for MovieLens and small tests.
  - Later for large data, fitting should avoid full collect where possible:
    - streaming unique cats/texts
    - approximate quantiles
    - sampled BPE training

- **Persisting tokenized dataset metadata**
  - We save the tokenizer.
  - Later we should also save dataset metadata:
    - source schema
    - shard count
    - entity column
    - feature columns
    - eval time/split info
    - version/config used for tokenization

So the notebook has completed the **flat tokenized parquet preparation**, but leaves **sample construction, truncation, batching, masking, splitting, and large-scale fitting** for the next stage.